# Matched Frozen Control For MERT Fine-Tuning

The first last-layer experiment reached 16%, compared with the earlier
optimized frozen result of 36%. However, those two systems did not use exactly
the same classifier-selection procedure.

This notebook performs the strict control:

- **Frozen run:** train only the `768 -> 64 -> 5` classifier.
- **Last-layer run:** train the same classifier plus MERT transformer layer 12.

Everything else is identical: clips, folds, classifier initialization, batch
order, optimizer settings, early stopping and test evaluation. This isolates
whether unfreezing MERT's final layer helps or harms this training procedure.

The earlier 36% result is retained as the best optimized frozen reference, but
it is not treated as the matched control.


## 1. Kaggle Settings

Enable a **P100 or T4 GPU** and turn **Internet on**.

If `/kaggle/working/mert_last_layer_finetune` still contains the previously
completed folds, they are reused. Otherwise both modes run from scratch. Every
epoch and fold is checkpointed.


In [ ]:
import os
import shutil
import socket
import subprocess
import sys
from pathlib import Path

import torch

assert Path('/kaggle/working').exists(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), (
    'GPU is disabled. Select P100 or T4 in Notebook options.'
)
try:
    socket.create_connection(('huggingface.co', 443), timeout=10).close()
except OSError as exc:
    raise RuntimeError('Enable Internet in Kaggle options.') from exc

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Free disk:', round(shutil.disk_usage('/kaggle/working').free / 1e9, 1), 'GB')


## 2. Find Saraga Carnatic

The compact Kaggle Saraga dataset is used. Attach it as an input or allow this
cell to download it once.


In [ ]:
BASE_OUTPUT_DIR = Path('/kaggle/working/mert_raga_5fold')
FROZEN_DIR = Path('/kaggle/working/mert_matched_frozen')
LAST_LAYER_DIR = Path('/kaggle/working/mert_last_layer_finetune')
COMPARISON_DIR = Path('/kaggle/working/mert_matched_comparison')
FALLBACK_DATA = Path('/kaggle/working/saraga_kaggle')
AUDIO_SUFFIXES = {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.aac'}

def counts(root):
    audio = metadata = 0
    if not root.exists():
        return audio, metadata
    for current, _, files in os.walk(root, followlinks=True):
        for name in files:
            suffix = Path(name).suffix.lower()
            audio += suffix in AUDIO_SUFFIXES
            metadata += suffix == '.json'
    return audio, metadata

def attached_saraga():
    candidates = []
    root = Path('/kaggle/input')
    if not root.exists():
        return None
    for child in root.iterdir():
        if child.is_dir():
            audio, metadata = counts(child)
            if audio >= 190 and metadata >= 190:
                candidates.append(
                    (abs(audio - 197) + abs(metadata - 197), child, audio, metadata)
                )
    return min(candidates, default=None, key=lambda row: row[0])

attached = attached_saraga()
if attached:
    _, KAGGLE_DATA_ROOT, audio_count, metadata_count = attached
    print('Using attached dataset:', KAGGLE_DATA_ROOT)
else:
    audio_count, metadata_count = counts(FALLBACK_DATA)
    if audio_count < 190 or metadata_count < 190:
        print('Downloading compact Saraga dataset...')
        shutil.rmtree(FALLBACK_DATA, ignore_errors=True)
        FALLBACK_DATA.mkdir(parents=True)
        subprocess.run(
            [
                'kaggle', 'datasets', 'download',
                '-d', 'desolationofsmaug/saraga-carnatic-music-dataset',
                '-p', str(FALLBACK_DATA), '--unzip',
            ],
            check=True,
        )
        audio_count, metadata_count = counts(FALLBACK_DATA)
    KAGGLE_DATA_ROOT = FALLBACK_DATA

assert audio_count >= 190 and metadata_count >= 190
print('Saraga root:', KAGGLE_DATA_ROOT)
print('Audio:', audio_count, '| metadata:', metadata_count)


## 3. Install And Load The Experiment

All scripts are embedded and checksum-verified. No repository is cloned.


In [ ]:
!pip install --no-cache-dir -q "transformers==4.41.0" "librosa>=0.10" soundfile pandas scikit-learn matplotlib seaborn tqdm mirdata joblib nnAudio umap-learn

import base64
import gzip
import hashlib

RUNNER_DIR = Path('/kaggle/working/mert_matched_runner')
RUNNER_DIR.mkdir(parents=True, exist_ok=True)
embedded_files = {'10_balanced_benchmark.py': {'sha256': 'd813d1c89126d9a722781211c3f851423783035c3c11e21f0b6567907fccc1c9', 'payload': 'H4sIAF6aK2oC/819a3PbRpbod/0KLOaDQQeEJDvOZjjDqXJsOZO7duKynd2qq7CwIAmKGIEABwD1sML/fs+j32hQtCd7a1MzFgF0n+4+ffq8+vTpP/3b6a5tTudFdZpXN8H2vlvX1fOTMAx/yMqsWuTLOCjz7Dq7ysdttsqDdxcfPgU3bfBqV3a7JqfHeV4t1pusuQ5WdRN8zJrsKktOTj6tizZoF02x7QL4VVRdXi3zJRXKgld1mc1P/yO7uirz4NO3QbOrkuCnLrjO820bdOs8yO+2eVNs8qo7aTdZWQZ5Ve+u1qL+tqn/kS+6cZuX8Keoq6DL2uvgdl0APPi43C2K6irY5FkFf1e7Mqh33XbXtZOTk3EwF6MLsK+BggFfuiYDZNxkZbHM8NVpl7dd0G7Logvm90HdFFdFlZVYbnEN5fPNPF8uoYk2WDX1JsCOnj8Pyuw+b9qgXgXzulsHf37xLtjUy7xsoQp9G98WbR6URZVnDfZ3nuMn6AsP9a7LG2ymyncN/FmUWdsWqyJvgtsC4EGl8j5ou3q7haah4qIstuMyv8nLIKuW3DvxDIXG56f473NAR9cUC2xpky2aOnhzHgeLulrtWsTgJsOveRsH3fjjzxdx0Bblut7lXZcT1NfZTZG34x/qXQkjhrmtG9Hrq7zKm6wDhL4DOljWt1XQ5Nu66aheFuCbss6W2RxmByoB9QSfiy0S2skJ4S1NVzukqDQNio2oWdUdzUF7ciLfNVfbrGlz+Xy1kL/WWbsui7l8/EcLsyl+16381UBv6o18ate7rijl064qFjBDMOuZfAU9XAE5cQcXdSmopJU9fFXvgKibOFjmqwyGtCwWHRfeZh32RhZ8D4/8obvHCZPvX1b3amj/qOdG/+FnU7eqJzAz27Lu4G2yvcdfQdYG27JT34vG7Hi122zvsUi1la+2MHR4gdWWCgF5Nq+bCl+2lUJSC6Na4rjp/Uq+7upmsbYekoqqVpUYGr1DlLYJdkYO8jX8fgtzj4j6lFdt3eCbNheoaq+BwTRVwkshpVUiq76tr4q2KxYf8isgGqRRu84GFveqLpey/CegWqcEE7wsEJ0E8N8rSfHvkODvXhewvLP7mL5liwWsuMV9StTN7yS3SH0f5dJcEKmmTPbik2wnpZV1x2+XtIrSOa8iE9Tq3HzSi898C+s4ve51ZGSPeltsc8Snoj/x7JSCvjb1AhGrSfJtNs/LiwqXAszXxw7Jpll+XGRl3oh5/udyI0vjb/EW1lYLrHmDbE+S966r3+QZLuuLO2RJQCIxvX2Hs3xycvLul9cXbz8G0+CBBhdC7S7984tNOIHf42y8PUUJM745HwMHDRkD4YJlj1m26nbZuC23p4ZcElX2Jx9fvnv/9iL98PLTBbT07Nuzs7OTDy9/fv3Lu/TjxcVrePftM+gLLOKAmEsKXKaNRsH4b4rfJD9nm7zdZot8Qn2glw3UVAVeNlc7lFXv6Uu0zFnyweRP03RZL9J0ZNRMsuUSm6EqUTgeL3lJhIqXTMOWBGm6gKkC0lrAp8W6RvY8vfR8k6/WRbXctTBtRTh7tMXxut7kZpv48rTJbk8FNCXaw2FY9IEmZjy+JnnOsJu67sSUEdmLNn6uq1y/Xefldhq+7LpssQbpIfQB1iEChICLqAORjCQKbG2eN6RClEukM5KF/+fjLz+TmHn3/nmAfKsVrR4cPmsD42XRmONX402FthAeBAIdGmNPoUXk7PkUlBwN7cXBuiSi2zHoOATiSyGYWM/vFuVumQs46ktG0moaZtstaF6eqbicuRPxAdFeIgsARhMIuH9B8bNYE5O4qlDmI5eEToCisN1VC1h7pIDhHCyAio9Bf5tf4U8eP6HCi4BvjwECf4FIlmoSVqBpGDCenyVnB8HMcXTjtvice/vw7GDldZ4tx+tiucwrIKaNH8KL7x6HsWzqLRDdwCDOkiO6UTYDtc/z8fPHq4PkWqz9pHx+dvZ4fVB6Clg/fiyeH+5/ew36q9QSAYCkXVBxQSfsml1+eCWC5FnkY9SC26+urVX5QyCaHARMJSGZAkPIEFQ90yVoYBFqgRNS/uIALIpdPkGNjwQLckEpSro1goGuJJtr4EcRP7TTT9BoDIsQVKC0vqZH7gKxPapXw8qOwlvob44iG/o+DXfdavx9OELVbA1LshTt4H/YtYS6Rt2JRYEYjDOgXyBUhNOiFp61i6KYvsnKNpfj0vZYiopNcRXhsCceGUkjRGX4su1Q4Ff3s4mJuwfNvKTcIzCsN8KjZkshaKMwN6QWoqBnAZMWFTDnMChWYHmAjdA11JdYfUc4KQsgQvUoANMrB52C9WSDF4bAw1Pm4aIP6oVRiJl1CgigT7Ko89qoIFjnUoEGtRfMo8jpriiWCglwORuNzLELJkkNMJMUDfe/9Kulki3adeRrs0K22ZbYiQ5xbKhLRhk2naByvoQyhvpklGELFz6zWsdf9oJ8mGbSFdBo3mwbYA4Rv5o4pELUA09MMkTXIPKnmnhbUTEmpKbX+b1cK20OlJjBim2nURijVjQJR6OEYURyZZikKMzGpF1nwKUj0dooWed3y+IKTP9odDk5fzYTgwDpB/Y5SIplSlIy6sBSn2Bv7V4vAcegA7fUccOyTBSAKPz5zX+8RlYJEKwuhWHyj7qotIRfrLOGvB70AwxvDT1BeYvaUDRSxWFRYMGkaLMSaBkUWRTMYEtbHQEIc9KqIiw8EhKbRwlGU0YrqIIVHTmcC8YY/G7wLzDhPwD3ByN/vQN7bNzkwsxnNYKMA9bnnoCtCBgFnMgGQF3Ly2WboBtAdJ1aQ3+RbsHADb6UJQswW1DRXeSSm+E09KrQtwQ+FVtABWDxMAykRQMIoh0IDLEehYgOJKqu6Er6QUPEH03GC5h+hEZ9c1QJLP4IgDmfjb5CL7kfl1BsNpK9tnojUFjIkdG/IH1soCs04oH27KnkuiO3d1R4sFP0dRhhJYgnB2FFl29UB4/pF1b46l6ZpMH0SwyReGpEP4l4YWpZuFCT1pq16FnR5lTJFmawaJ3yJylUJFJwbRmIkcV6xOQSMXIuaCYyycckI4RtdjqYToOe6ceSjWG0qq6ExX00qBjb1D1iqM5syCdFrxZvIeydeGYEPwxMiPCdsaRqIzGmg9OCZHWpBQP8M5NKRH2LWLucqaHxfBfLmH/hMNExIdtJUKeUbY8SJDb4i6ppu5gS80LTRticao6Neeuae2fEWHLaozN7LDZBZ7tlUaeotHkIS3+0SMvAOrVIvlQNBx/rNiFFkLTENtJffUwGEJewMRj1PuJ/D963WgECFIc0Q5F8NLSVXhWhKNmKUa+UMfSJGk02b/GvORw/iH3vrcYc6Fb5tgsu6A/Zpy2+s/HCisgq/Aj2x5Y3DKRoAsJ6kOPcT4IHqLu3lQdEqEPhQv80CJ3UULYCHiFsKUlx8wMFPJggjkNkUW/vSfsHQ1H6fEGqrnZlCToemOg3uSlLkS+pLkgSMflRhnsQH3ZVB2r9RdPUDWBiwBcDhkmO/rv7YFnnLYEmgIAY1QSix7NEFXfBiXSWLlaLgzRmxw2uXKCB26y81riL0d1T1rdlUV2zoueIG6xKjAxqE5ie+JUlgBPc5k2EGuGyRTxGoGuBWhl6Vovda7lucBoj7NUoOFVgR55hJqibRgIf2Evrs2JRdiWXJ2X+uTiKR5GRaIFna/EoQ9FFglS/kZNGXHjk44hKtHn1pwEx5+FUDO1oseQwyomXV8xBM712KQPp2F8F/Y5FtctPPGx8ARjA/cEcqVlYdj0AiPLeS8SHJAB7coQLAKRTg04A7+CoYNGmSHhCr6dX7W61Ku4kdSPshzDZbJ8jjmE13dDfVZmRozjZfJuFNuMc+dDiDvVYFOV3wLjSFnVAWOpfjha33UFEcBNTB4/4tldFgE+kFrUKH/q19ogyQptVfzYsx/VQL89m2C9j7KSWuWOBYo7qVIISd2PpYNQf+SHtas0JRwku6pRnOwpDZwkeEu9+0W6J9WyV80KTbSdZm4KVCU2NBsTv4zLelu+oOugXMMi2Lm9yL/ghAv06qa4YqWJo9vwb8t0UnojSg+LS5nXhzzVbvdAQ7kqc4o4Eq6TbrMD9irzJhQkE/88bU3YmgU12QhCTNiDYv9AugydCDrwSNkDwbtfCv2Jb9QnzhRpQ4ILsb9Cgm7/o2kCaE1rOJ7rukNIjlB3kRukC98L9Go9WdqBLQtfh4rjxJyiEN24mwZlhZ+mXeyVG0/9hbYEXF3SMJL2S8A6LdfmlqHUM4/WxUcTFpYWIWfDNNDi3+WqpG0JDkDWXYXAOFg2IYia5oFRg13Xd5qna6hYarFLnJl7Fldet8phOcIbFPrXtGjW+2K5RARehCTNcqBHxiUM8vvYFNV019W6bLyePlQbQRqhGRK4Lq09IkD0fHxEA+14E3UT2GFC8XM5Ge0OrvcViNvsoPO5DKHHJTHQ2YjdCpfpiz6wY4qVRYya5PbwTfAto5KpA5xtI3oi4spgI7r/5BpsSQKVJjF0Eeo9klb9N3WmcWa2wjguq2LTMNvNlRo6fSRCNEQj+vjyfgZ1dsHyUa0fobRx0RSiX8C4nipJmkg0jKFl0FPzVoLUTm+C1JhbRjMWB7EU70rPHvfENHjraG8lYDuILZMAvVXkfPFjd3gdMJusMJH3WYURd2wnjUuN2L5DtCoIVmAPL3SIPjP1mpLjeFnK/4oUka6K2loVcj3BB9HyqtwKNUEYwkMn5i9neFALMC0CaTkUgVfKB/kTGbgBjijfPcTEZa8KhPYkd04Shb2oe+dkzLTQrWncxpocbviRSxW5WVwlvbihgDtIt0ca1BUO8baCZdNvkK6DOdScCeyJjfCnISrG9x6FDA9thIpyHN7Iklm2hGB/FZxXKHuNzojgGLbZHFHZ2JNm/KBeVCHIzWZXL4xiJy6K9xsmjqLoEn9Jdm13lkcaW3IS9RztSRf2Y+3xS4lvYMraYACIwilRoWUJKQXlcb9QVo2xWljTHXErAChkfkTHIZFO3uKe52dSVpYv2d/FoI83ctLO23Xh2VOEH9mhqNuTjQrKWZEN7EyJOoB6jrcPTx4lHsQ+7usvK0Gy4r16HxlYm4UNOM68tY56lM19wUhvUvmfsU7fM8RERytGp4nsLy1tGHG/fDwzUQOPToU3QP6I3RLhX8143FFob1N8jLJbQq+A0OM//HAfPHNwArdJeqVEe3wwWXzV5bhfHN77ie7mzKvf0OdxArzOoFIrIxxB/Ky7GOmMsl6Dg6Wwuhb9VYfBNEE5DQPB334/MT+8/XLx5+9OPf/80UUYIGhkcFn2TNyr+MbSqeUBpXTz8SGtcKOHBg7XkL58Y2vCT2Z4t6diQcGDCOzVshRcrKUeZGZAkbMNfW71LmYuIaeyG4iT70K7QF6Y9bkDy9EkFnBRat1GBElb4cEkAGDszLBnpPXHUAc50/sKwZGSnwLIn3UGKbafVj1LJYpWp1+ggGzrYkl6KUmdxxkomrxA/RpuHV6NuEogfaHe+I/teS6BHWJPhIjD5CQD4ItbhkCmTGgyeOusffYx6ldnmXvWBAvPbuKeZPRDTNQc62rsGt8T8G2QDt3VzjZ4MZAzoc7f5wyR5ttoHP/7gXX64qH/DlcmKjSlVoqNViuONsTAMf9xloCF20LscmMO92L0CYyVraOLxiANPPscmYCnBTDLGqdrGEMXQUUBHM4BDXs4AnzfEi+knntCg348rm31ac0yevwbfult9qPD/J+5kyx0SXgfQ63zZanV+BcxItshHVLo1zM/4NrvnQSSGv45HdUmjUGYcV74cP5v1C9IYeyXPfSUJS7MEj5DoohOCqjGaSu2ZUMuKygOpeFqtdtW8vaG/0A5PbwnzBGRtmzed247uWdEC9WKISxf1yhBCRv8yGEbXYTiM+2OhyAgNGq9cScplOhAG5ET0cPxOLzzHiOwJx6nY0A/hhxnzw2FCol0RyVWvViCJ22i5a0j4TgIR9+mEeqn3JCLIFUP9rLYJRvc3mdg6Esp122UNGm+bAoVOchYHm+wuOsMfsiXgK/D8Qpjyol7OkR1Q1gRkVBq7HYM3uhE/bECXAf6vU6uTvdAKGBGNJ7oEEPbMwaeyqMggc/qn4QsMSTzjAYmsyZmnR19mU/HREKjnWokDc2N+M8SS4TejINVJMK/r8jHfGIbRSXZMZ2SAkOWeheoUKYniY8uKocS4VUdtHKs4Mu7Jib3TaFX5sp1GMUP9HUbZH9Vnuy9OP4Tl2Ww64LuRqjOy5+LYIFuFmsljeP4S4cTeiDZF4UG+L2Q8rPK7a+OpGYkpVsMqK8pdk4tNdFuxSpkzG0JvQL3i0kgBNjVoKP2ix2LN8YDSHrdUkmhvexW+anJY56DKPOgG99wRdy+4t6lN2qHkE1NeOpE4K4f7wqn8SLHXUxJoxg7UzLNnrDbnWOcWHKy/EmE+nseeFdqHKNgy+kIGGHWPQ8duN/pQ2+yGCObMu3nKLdCOCLfl3wMXVlSKSrHAGi24wficHgaHQ3naZuoNHe7FFNRVzQQ0WITHwKcYIn44EGkksTodjHAe3ut21EAaKWqB9jqdDLZNFQCbIF22gEeB3wjEmbPSxwb8kb8PKPcfb4n+Xk5s8LOhQTEKoXdI1Cgd563qBAwTpOT5cJP+/X1FdnIjjbbTje1jIJknUot6Mhvt0/aBiHdy9my5p10xLzwhnzR30gE23vLtKiEfLS31WJK2yTO91RRfv9TsZ3YwLO5waBx70/T+9qM720Mb8I72/UhFufOuvbKPVOBlZBwKOHJ17b9gFTGP6u1cmhulzMWmPlVnsAv9yB0L2l+PBibF57863UQ5ON9a7v5vmWYr3uILuDefrCVtgAIUKCIEsHtUjSb/565oyNt56GTK15DWUfEeR8/tgQjXL5nTr5rPL5rLr57HMEcvheBFgKhDEbr2dJ89WvIrptkXoCu1ejlj9vk09mVbVgo+pGp+hQtbPo8Mz11+U+S3IEPQ3eUcolF+PPIiPqFZRhfxqXxDJ1TMF1qE7Sd9/x2XMTAIxU6ttxJbCJS12972hHRgyrFcTs6NALHjd53DjzXIYXaHSwNggfkFyFzbNvVNscxl/pSqrsYtCtYueH4mzs5yB3shSK/zDnom4paY2XZ10J+O5LeqhyAxG/0dZD3Jls0Ya8ls2e3qrTTKy7pLZZx9u9tssuY+Ot5YiwN9KtPd9+wdLHAd1tpc7hlVPX/0ZDA478HmNbHiDKaDey8MvoY1rO0ywfCuN43QsASOtmWXrIormIgI/uAR5ml0fh4H0nvSVm1C9gRijYKkpgQxDu6mMgp3vcunokdxUDfLvJmKDXgqKjuU7Krin7tcxW9g03ddgU7kRuRpmT5/gUdKATLuNYW6HB2YilROIRnHrE7bT5j+AmAm+qQDd8kEgtvwZXZf77po9LVnZxESUjKeX2WyW26L6fl3Z/rzoqzbXB3npcMjzHvpoDj7/ide6pJaPx2l1H42KqqdbcIxc4u9qJuNQ3MDRDRkuTHNCKmANpjRg1jbWpqXqGYlORLkJGvxzDaaCKQXPn82slahqqUOA1M+j1SfmWaOREdA8UgmCSCWCeuVPtsTf6FvRTjSMHjTdaTRbKRI80c6yXozoMAeSzy8wzrxw9ThLpuCUqqkau/CZSmpEwRwgK9wH6XvTnYYbaPwQSF7n5q+lKTafg6t46AKxqO+PHuHFL00Cw6c1RONW40K4N4JQhalySAm+tRFY9z2qW/TLfCMMneo0gjeQWNSD8agsHAmUbC8DM3XR0DhPVsLgnx1RG2xyWnWlq9Ohq13mwzkcot6FBAbEzSyA5BtED13twxY4iPHN8UCRQXnYeLHKFzslhkdlOfX+IgnCLIbkN/o9o7k0fjFdifmcomcQAEidnD+HZ1FJJgJf54GAjZVN8o+f2btXv5WYfIn8vcJRrAPQI1/YGD7IHqg9vYj0brl88tl2iDoji+bUILHizFIi9yvILJEE+gG3bUYtLWpyaW2dAmO80xNdTaiLwUV85hT6v2U/lWxpIctlsdjCF8p3U2mxMBNIvqrkJgEP1UiMByzNlAww0+4T16xHzDsxaZ3ObDFa1Ddu47y1SELaHJKvPeLTGtH1oMMhteaGx/jhpeaySf5TVZGowQPKNBUjk6GmZxJ4sTwBuiaUuOxnM2ljHWlpxMnbGlgvHUlHc9NVl3l6IszIqEMsTFSLmmDk1peadcfzXWZbU95ZJfc5EQ0/Y0B3+Ysprjv6RQG4FgTPccuIp5QmjtB55gBo6UzKKJ03+JVTcYeb7IBeTrQYuw5vEM8d8B9y1wp7SjJWzsNt2b+J885IxxCyufXaScER3QZmq9B7VT0NeU/cdBbcUpH6qDpjvOtUSwig6QDY6H9MeyF8DuVCz7jaGchGGzIftFbEwf3M2j7jPlnUa3A0qpEvpNo5PdiMbEqAniwMTax8ArGhQgL4xxFuOfZUSmcwv2Qm+5LkOHr1qWL7lkPSV44VidxKwZhRk+fmrBHiVXK5y0yZN0vu+6X1Tvg3c09sdpB/5FRJ99su/uUOJbnHN5j7FtZzz++/zXYUMuwErHVdbaj3A3LXYPyUFvfSpmRqxmQ5BrjCuwHZNlMNWbuquD8L3RquMyxCQ6moSFgmrSyBGML1icGICZ9uAZ/d5eFPSMiLwdJI+Cp1sdR8G/T4Pz55KsRBnxYaQjMSmAg4VDRs+HekUbS799eFA/4+S9DCO4wca1OasuhXC0MzYe5g/jCCB1MLbPN8Rfi59+/+/4PRpAY0xIzM1EW14cDXdh/6aChv95Ru8O2BHc/fYtXvqNhdMlRRinKbKvfs56zjOsWy7tYjhkPy1S7DWWedea63/QWTEKyTLgg66qgv2CG4AiwNz0fJaADwxvKngp/+ybx4fFcqh6q/SRu1EDXcajyHsz/uTaTDQtGIfIpGwrMqa3AOFv5BgSy0QBdi2ub2lDXAq60ADxWiNjFelfh9n0GpuP0bCSikvAdToA1GscvzTVij0OTO8MWGHdEhO0cjvGcSeFfzzH5swtPHSBx4TkxbcfBROyAAvUZ42+3mH7WPeRtmLa2EFI4nrpItwvy+KcmMmJHX8ABTc3R+bCpjruSh9YrBAaN9qOMbncYJ0cb2+bYTo42ss0BS+u2ZGWAz/stEpFwI1JRYoP2rXHA6JCI950B2mTXeASonuecDlbm0bWi++RLTRz2ofcobDF7buhm0+1tDveZfqjzfod9dbufHdkvNl5Nz5OBzR2Cn97m6FqdhvLwZxgP7JzfoYnSTJ+9OBuAR7vezTQs56ur1gPGGbHxOFMZyKhPYD5zBvRXCgNRVSVgqe9KeaIXZygFpRBIJcVA5lUsVF/g5uwSlCJFv8Djg9RALg6rBiLvptgPN8OkdlvKFaKaMNgNNJaAnY1B2chuquRj/s8dKrdgElsjhE9vkUP+DGZYpHrnoAELUfZrXcLser/0jxdvf436r1/zUCIxpMFWNGgLIaPY3apBDAPDvAWaNREMqOOlxOm8aWmYL3p+KhNfkdDidYwlelPJKuATK4xAechCG/6xcCvJ8t6P9TybF8A8itzzuTf5roeYSEA4hynRtoiCfW7jaSCXpUzKrWIr7CzdkeDw5hCs02a9NOMK0EAC8kchUpL/dHWuAMkM476aIOFh7WZX+ZTrAcP6nEP1ZXFTIHMB2W/CRvw873fVll+eZOVG08ZkxcH1lErHUiqSDCcHjol6vQZHdnpJFvEWzOgYmhiiMxUG3v/ESSKbiZ0rnWmp24HsNbxVnl0Hce6Y+mpsDsjXSinyJDijqE7em2vzTmXhArToJSc8EkZ8v344cU6Tcx+Usmpi6hLhzFg7Fprf6MRSXLTnhrvMNXpN8EfZhsBdotLFR5fyeI8JdGSffBpdns3gfzP86wZ1swZrDWgUaw3Q7sZIU0tRpfKWjVSL2eFdLK5y56UU+HDf/wBD8ZXH1/fHMKfHzxTz1m8vxJzzsJpvDdrsC1ZXMeltxTFbnJn5O0ilQa+8q9MYKKEyaHFx4WTFOp2Yd4HPQTML0eRC0LUJt966RgfkUcWpfe9E5CiE5Phno8/suMHsBsrdG+0Dar77VjJIHoG4FIU2LxjCj/IN2p1ZtcMh5rCYewfoGX5Jt2ZAbX2FRmSeltdD1F3V7ucpHa34Lpbnm3Rpk4+3691qJbbj9FvV9an6JUf2B282ccmDyPhi5d6AlmZl2YcoN388SuaJ7Z1Gx4CgVOFFOTeT5itFakrHCjENempoV9qW1It7avw2UvKzymaAEW8E2k2XMplQ267YwCRr2qI3yctltvmvyN7lwggJ4Bsw1NZUGsvGaK1s9Ac2B9JlvsjupxghbBL1AkNuGw6/B43yVVO37UXVQWfv38LPSC9e3gbQ+4h69eil3d8+WhctZRlU8m+OAUKw7q/N/R56Sa6e3lvKYq/i5KEMHu2hzPb0UslULkfnbVHJOI8DjQ1R/pvg3BSshEsiBsMUKGHQGIaEwJMzZ3Ua2pUekLG+HVEvu8Zr+E7uVVH2PJMj+DalMGWR+OXSil3wXhW8HyqoiCshBfCqyUjfSLs6xbPNnq37EkzRTjvsRUfcMi2WUAQUcSU1yn7pZA6inYyP0UnfVQ/Ux5cNUTwadjLF/Dpp5CF6sl3xK5rCQ4Ntu3wb9XtBs/uNOmyC/ULHEZjvwVPird7+9whAHsjkUYN2coUh+cLxuMwx11fkeCB9EIU6JYDJiRyq7xAvb9mefPHOEy5YZ4r1Ch/1itpkrys7g7b6euKsDWk04L6RbT+YeIj7eNbdwYaHwZAaFrv9tWtL+wmPOJwfrHeE8WRIDWJx3kBlTzoPYkZgX9FfTyYPyRlayoMiCfbUI/YHK/esOPu1r6KJXFXNfDlUqWeTmi8H04sY9CFkgVUvtiY7DvguEytvtZYjxiYWPf9Nf3M4qyF68E//oxQ29Lf/WUqo/qxyQBxncHcW76KEznn2HfXha5EjvxKrmlpJKaHAqJfapB+E3ROJwxFMVtHesQ48gGEW+NvUEKHyMpj+Rgyf5jgxp4Xx1E/6398GCT/RzZDidkTjWsRlwQEz0vvNiQT43ijqnzx+zzijCAwDcbobZiktHyXrHA7KFat6JA0z3DAh65giTndzjMBtUc14hukLZawuPHybyGhdrICJ0ihYVwTf8uoH20s8G6t9JnwlU35Jaz+0IWV3N+Sa1sSKR5xL0OvDecmXLuF3sGLABAjHY7z4ptyuM1D7XtiQUP7fcdK68IL61P9+L76TUjjOWSv0dYuUCY4J/iQnyS12fgweFAdycaE+HA/R4mganr4K9GuBKo7nBSov4nSA/kEzd/7IzJ3bM/cRRZz3e7HBQCrQnp7ZX8v8CoWYInsg9S1PrRVWNfEuWrlCH4nyNkK2DY9HLwZWuXOIAUnYybbC6xCGo7zpTAJdvslXI7XCCaK9KI+0JJwTDMMya7HXWr47+aQ09wFRqPmPkxVKmaFQaNgQpaLa8JSZygbtUSpvGKJQwWuWckYscTOZCdSyTrUHWxGrHJKjtBiJto5Fb6KiyOxjIVg8DuRkCe4bG63KK3voBl2fZ5hATDxOgNhwcU1c59iJvcVrO/K8aTWUs6SCgXmsk5Hwn+hWvS4v3eagz+wohd5R5HvWed8yH7n7OYK661WH6ry04/xqvT60w6ECwEJhUTgeVhFG4E2AKE5W0MRSMbFxN3iOxyuedQPDJ2f+bApjLGBydgJwQFyENfLgR4XGMWBNgRG2frCO2ECwlsgguMeLjBeyjZXKY8hzEjxocHujMSlR/s5XH7IGx4PRhXxiRZ8HcgSEcS02UOPnvOJrsS1wrhTCt1dNsYzkQIz3lmQ6QrQcdRqICLlrAcsDLMC3oRTQeAWlD5LtNm9Ac70runuRg+J5TJuPz2DEERpzBgMIxsH5KDg9DZ7Lg16Sbbj+eMfrbsDgRjHzETIDvDpZi6qKgl6ga3gc5plm2rqPU/0zNnymRTcNtwvzgj26a1hFGofZrquNr/JaOSSfae9WObf3gjUNr/KH8A4kDw/qchIHZ7iC7q1X5zN94I7naj/MEb6Pg++Mo3TQftfhuPuH6e5gNd1PoS1xoE4crmunuLQkdX7vLgH6t3ee7nJm0Hz/lSDs+by+Q5dcVi3WsMYjXBUwvBGQYL2YhrstnqYr81UXft0SAJaLLRQVRnaBXu2c6bPXBZpHO9RnWKgM7qIh1gbOT33hltbh3bHBKygN+EdnAlJtH1tD7HICdZpbxLwYiXCUHScipbjaPSut6E/obZMeqgW4P1CH/G9uO8DTDzXDLN+sY6iHIqOirCzepkKV1oLW9uGbjnornFP74M0xmhFoQuE9m1lpKUE2EKPUIVr2xx6yTZBGrGYs8W77/tC5h+yFQAlPn4HRAWCGqlTf9jw/QtpPjFjWk8e9ao7jkuZbOyDdKDK/k025L53aXxb0sXcO6pinmikJveujo8QFrkqjXprq01ihpOe1G3DYYZhodR3O+j47OhRBHydUIfYgfn9iJgeUzchTDno7iZUgztPFrQqNTXJCR7G17RvkfrEHacZ+rVbW9B7QASPXaIstW3Nb9o4JmfmBE8aroxMH14Tuy4yvsvOQulHGsD/NSF+Ll3m7cik+xsy7ZtJ5UAk7zbuujc+0svVYY6MxYX1jJ3FHzGBnQ6PQFcSKNxuS654Bjsyy9dxfli1Nq0ZJYXeURlAGuRFn6UW+MWOOdXdi3ZoTgmYEKfFX/q22FAfjoawhGBvQfcbv/SijdPXHXAZA6cmXjRM+dU/k/g9Ry7mFGy50EDn2PQG9eGndZv+dOdCen2O4H5fkIrGGLTiBShkr44pkblVSUKV4fUQXN0lkUeLRoMbAgJkbvyjX9S7v6Hpm5uf6leDrRsuxEPBWuN4yuwESSOd4spQSFTIc+/XjsPY6MBSVRroJfEehx0OBmYa/z7ni03b2L3ALXYEDgQGIuFMrwtpqOzJCUDjhXRe8qdz/u1H8lWz7HTX9umi3wCGixQY9G/RbTPa0p/mwRe+cfZhmjnRfbLLtNPyBTunZX1jvT808GnZNtOLnWcMXsvuOAGR3hlPdTcViWtkPznTsiRloxAeM+Nhn/fuOHviNCtewsHt0QLz1upe6JGFIPcWehMEy0D/L0+tQrcFpQ2YB8gowbenbbPPuSJ7p4J2wbBwE8eH3OA0A+9QX/aYH25BKB93YLLcsv7UJRjL2QyBEOIKCIU7AG/7bWD8qZy27pXseWxYWhwIzLcVq6lGxhIt8epSO49S6n0q9xDIEJLBDqpBdAwGhXnJ8+BcashSN5SWBqfHb50Cf6p+mSBOeb1Nf8TrDH5suU07ZMDnTh9OMT8qLMo9rQGp0gvFrFDoND305QtTLCobyZL06TocabvYPVaWczloYP6hYWYWOw7kHC7H/o38Gjte2fJ07rG15+iYdczZH7xEKXjBjEHg40ifKrA//IyfLBo6KaNYhN8ocEzDUG3O4HdJ3tlhFLd0xnLjapFFWmKB5uyvpLiSfSSoFYe8GHHP5Usd6Zo3jirDmV9ewXscn3hNvfJ2MuHWLhuR53z/v5rINx+Lx5bPsYPGIK9v7ql2/uOMe6RdQ1/AsOk+ejFE8cAeSSaI9zD+6sSo2gEmkQhFLwh6cRA9nfmQSfWv3/88kDsuA/y1zubcujzPHp7d9jjnY8aUJ0sSCNku7G6juLW+0FWjd0Bz+KVAZ+t5dfPgU3LTBKwALqgc9/iCzAxiKamj+/oQ5BDA5xQ69xBmy2GJOp/PL+6Dd4K0v8x3dl3KdXeVjTBndSwWYmAAv6PYYNlCAQDG4CwBTrorMSncZdOuMHHjZvMXEllRExX9hoiO9dZoMdf9Pfwo+AqfeDnxfhWN5xe4k+G++/ElEVu7/2yn3ga825ULqysq9U+qjlbBTJj+U1dzrIZ3Kr1TGRBMRk2C3xRydDwPXHPmgBPr+EKuauqIiOV/tA3ktAeD52bfB9d8/O5AokOtUo/lUTZu8VUtn3n5CU4O5wU/dLwDA+x6h4QdzcgCDdMJ9V+EFcbXMYcXpaDkaEJq3a/wA3+awCPDiDdpl/gsUZHKvG4vc+RgEDLjJAamYmn9pg+LTtLxDMNH3mYIN7dmnj/EiZ7xxuBNleKvnG6OsDV2GpRAvF7E/Zk0jkwOFfWg443mG98xB33DZdTVdgX2A6D8IbWCgxO8B5UcLfg90hAw8/IDTy0bs70xGKtZaPMtTq70PEifyGbo4fg4PfG+YUZpf+ODwFwMQvxCQrO6Px2P5/8nR/wgI+gimtjUFs6VtK8acPxus4cmgasYz6a90kbCTKUBoXqCzmsQVuokCHHUBiitqYRkZsKZreETsXRIjBm8qRnFp93fmlIdpmhrVLm09YtZPW+EUt7UIuzyJosG83SugQDNB2++WD4tQSS/FMJ5odRozL//uSXizCh+w95dPJEE9mU2S5ysBGj/0DlzrEgfAyU2uHjj7rPQjsAhT3r6JL1/aOVHN0zvxZaB74Ykn58+COaQiGzDbxKtN3nTpn19swtmlpOOZSyaX+pC82BkGVmXC+iogQBpdhvQm+jYmsFrLkadlBhJ0hI47EXnihcrI9bFrdgsE65ZynzWf/Kic80G0BiECq4ySK2HsyAi+vyan+/gHdroHEV2MbRXpwbYZWNjPm/F1fEoYipoJ9IxJvVKHV6l3hRKYyyd6o2KISmn5iNL2doSHEmWQwVdO68c1KOQg8Cgh6SMTugrf4NH7dS7M1kA4VE0dYVNUu5a1B6TRcZnfAAUoOQVT+vTpA5Hn5BscytOnyWNkREo0arJ0lCFbdFo3jkHVow6JmW3XlCd0jrpfjvoJJbfDXO/VIhc6MELjPECg4yElgqZcAXDyESOHLkCXBnx3dCnjT9WygM/MXxdQYLODv7288HgIAw/oB8UGLZwMNG6gl3W9rDEWFOst1jXGvELj2F/iMwI5Iuf57bpYrCllPyqoYK2sKF61k6ml6hXVbDHIqwHVs6HFyKo/XSSJj/JcJGn4qCE+il2ggbfFpuB9lhZMo2XwvoaxYvbU1012OyfH02OEMQ74lnevnq67ayY0RyWyRW8UTKFYjqC+3OP9ftVVzipcZqCBZSgrlY7RIhTEl2YCwbtF3my74Kq4AQNpU0Nrm7yswQakS3xgoYh5r4LvnTox2WSZPjhD/YWO4x4lyhmMXwZYYGzUeE3nZgfzVtbQ6QZLFfWy9XTOOqZj6CTBOhP90xHQ3DV3odGcLuADtgpqsyBlsiKF2ZUEPwEWB8y8gGIkWmORLNb54hrzH3p7O5eGAc0V2wZiwRUt5cNrMMUepuTNOVAEIN8CHee0rFY76Bcsqnzc7ag3t9QsLJAGBqBZArCUZbYB81ecsRdLDP42OWYZgxaYNj3d/KDWAU5gIXZDxVShBFliCqIOA07rCqYeEVJULQiwjQRLGG07TAhZVKtyR5zCPK+J61FSY2/ZJgOCh1KLs88DaS3S928Qox7FQe8uQBlPCy0AMqRH0rjDWjhLYDYWa6BqZ+8Yv6nrZt8zELCN0ODZzSm7MN1y8RktJgLZCoOQ85NT7t1TbUKp62ZlayLhunWlttkT6Rs2y6vk6kbWfvPzrgJsXEu/ckXXRlsOGPsC7w8X73/58CnZmFanXUJsCw1+lzJ88DttG1mmDs3852KLN28l/7fYvoG/kTkKsDVuMUhMFvnpffr64s3bl58uXtP9jqLswMUYPOiJe4wQv6NvHUH6st4KoOK+LyzNeaWTJi+BrKFrXW2QDp6PSLd1W9y5Z6jz0mgNQzk9jWGHZZ5qbuSqrOdR+DQcyMdbrHgBHBiAfyCMT3Fh2HEDEVsJ5oyoXHWYl8D28aF8wj0fdDGm+CBz3VHUdeJPt1Ftk4Pfj8jYQbHSGI6wBqsgEgcOQOJ3OQbryx0YgxintKAjkqfm8NU9amx71BWQPOW8ltw45XdUU8WJgWiilIgioKS4So2XEb8aGTdANHXduYs9pE8hXUKn6to3iaoLGai+uKdI3PKjb2pwCxlbyG5Acq8PcoGzT5hXs7egXOlOvHK/HK/4E5fOzXN1sdmSuflvRt8izR4RDG3dBPDq19cvA7V9NgHzc2hjTR40OW7zTcL/8f2vNlC6EJSOMPHNhGcKMPEBn68TDzE/Pzs+V3/4MsA0z+j3Ftc9ccIu1Bm0eqa922bSfrwk19WsVZb+eivEO6uAII0/fYvKw/vzszPUJuAtp+Ux0vgPXSB/oZUXokWTpAFjxpPEj3FHk0F1SL3uypPXcT08fcovYiRbBRADzQzw+nYLQv41YYLyE9D6sK5lNx3smDo5bGmrIF1koFR2xSJ8NGduOB5zC2MEM6YVCGoY0ispCls0Xlq5BfFKwCU1wrys3e2lxa3cjy7P6l+oknUdKyFMB7J5Phnx4MKzUviSo58viEW+Lsqi0BEvo153zHh0Sr/Cu9upKMjGy0A1OyEAfVzXdOhGj1+99Q1cffyS64N11q5N0eBDgseKClDsP+eRSRSxhj/F2+50V6wQclT7eKlfF9tUXqjhv/LmtfgK86S2MvDmEmuzB7dAHlRj7g04/juLhcEiW4++8pLDo3PPC6LSl8CB/SMbBxOILB6xFDThC62CrPa6HsoUXuIeK3E5JuAnbYB5PRFPy6IFFveSCJwMOkHickoHID5xV+A7NInkZtsT4nBgPAee5YzmDR4SHswT/8jNJgM3nHgnUk6iIAyT0I+awGMmbyVnzjBk8ao9tFksqkuCV2jO8sYnolqylVM9zV53sJyLolUmf+IteATmPFhz2IzgL2rFmgtpmLE50Q98UwqyOQoiySu+Ac+xj0FSZYSfAwX3tkbya0vX5tAeAt5tSJ0Vu5VUUy5utbk3Rc9WDdq0csKLEVoIsAP79Aaw897xHDlfAatoLBmfTKkefpRdIsiTULuAqYJ5D7wo2Xf/SjwEwQO5rsTQuepoL2BIfV1dZ2am544kdHECnZwA2yZflRQVLEIgtBbKJBDbUx4z1lRPY9HWI8oI7ymJ67WEHmJWVNvHHPeG0ocCXYzZ4uLxibnR5cRZ+tRE/1fvbahUgG5Z47bNqRwcGW94iO7LoTn3Yx6+CDN2YojluodHkJXFfEdexS3l3jhxUuj7FqLO1D4JZMZW4yZbJrz6luiNiuqj9r5boXQZVaO1chftv/7GPNl52c3Ddwr01TQzK71YEVxS3OMKyi4dCuKGRqYCgCmRcvUFtVZ8VEvkiFsezMYxJ04XiJNv5DSltR5kK9yzwb7gzj7G9FN7Zkf4/g5iWA9OoxhhYkxz8HSIhns3cBiMRdRVs23SjXWxupuPam/iykQjIkv2+mvkpoU4cZRPRw5YGPPLuwuJsgfZjT0sIrza153vgTutHrkCtqhWNTLPFebmqK2LO11tkJtP6XR5K4JgG+xJ5LVZn1q33veupMHm+GIvisVCRBvlkbaohGjNmAbRgX/lwpqfKlJk0KW/oCs0OJRAXJSMo/dcr2xMqtP3ffD3z7D8jP7uORFAO3hJjXXjycPet1EqLtXBeXv3y+uLtx/7TEVDuVQVZ/r2td49qAdOgVIKwJXnhnHNuz3XjFgiSckVne3X85GFju/yERWXzPtQQ3hBkRFYoUZ9xMidexsrnmwAj2AE24qHjgnFfQitHyPxgdumbRGrfDJy70GKWfEoFD+Wq2n/buh/NSCI3G5GwI8TpP6vxeuY3R4MrHnwW2M0MFCfBybKdzfJxA128tewot5lwIHxcuBa+6dPHwYd6itWlR4w7iAUCRWHC8Mk0ISpBIrDAU3eRIqeyO3jO8oi8o/oqRNL9XVdHQz46FfYewSelXbFJDdMHJUu2pvh9SZKJ1AoxJzyy/yODzaaJkTPcHC2xIQZYei7at1aG3kVZ4mJjEsxMMSIPInPz1q1+D8X27DnfXQF7RSdw1a+TBNcn6MooCPf5uLw3qe962mZfb9Vb4qqaNe4j217eT8QxkAbtHD2ROHsyb5Xgdm/3LJDN5fuoF368HSjRVFdReZU4gZvgffz4AjSlO4LTlPcpEpT4bnlHauT/weYmrFzhMYAAA=='}, '11_crossval_benchmark.py': {'sha256': 'e525b1b436ffe44eff8ebb105c902663349c814cf9c874e763a85cf14322bbae', 'payload': 'H4sIAF6aK2oC/9V97XLbSJLgfz0FBh0bBt0kLFrt3h7uci48tjzTsXZ3n+2+izsNAwGRRREjEGADoC21Rg+1r7BPtvlR3yiQkqd376ZjxiKAqqysrKyszKysrK9+92zfNs8ui+qZqD5Fu9tuU1dnJ3Ecvyk+icm6LlfjqGvy5fXkc9GK6N35+4/RpzZ6tS+7fSPo8VJUy802b66jdd1EH/Imv8qjV3lT5V2xTE9OPm6KNoL/dRsRbfOiisTNTjTFVlRdlK870dCXdpuXZdQW1VUpJu2uLLpoV5R1l0bfd1ADcejaE7G9FKsVFGqjuloKavH5i+gyL3N4XEVtvW/gdSOWdUPFxgi8ipq6yzvRRuKTaG7N55Nu09T7q02UV9G+agWUhFJdhP1Oo48b2eFL6P9lXUH9RlAP1k39q6j+BXAob6M8KotK5M3JrqkvBYBawSvuDuAtgA5lVIl9A3+WZd62xbqAPueNQMJCzVWK9D45AaDbKMvWe6RslkXFdlc3QKKqQuSLumpPTtS75mqXN61Qz1dL9WuTt5uyuFSP/AdepPuuKNXbv7Z1pX5v826jftet+tVAL+qtemo3du1fi926KAXju6zLUiwJO4Xwq3pfQbf5+w7AQ/Pq20/YGn3obndCV3lXr/al+Aiv9EcYHfX1ZXWre/7X+tLqHvxs6ja3+rIDlsHu7m7xV5S30a7s1Pdqv93d4rtqp17toKPwAsutdHdFflk3Fb5sK00SYK1qhR2n92v1uqub5cZ5SCuqWlXcl/a6BOaoUmaSbFuvRKl69ra+KlqYJe/FVSOAM2qvzlZ0TbHUZEpOIvjvVV2t91j2XQ5fb14XMFvy2zF9y5dLYLTlbdYChwt+pyZHFvq4VMCyLUHjt6v8UyHa7LLelzBN7PLrqf3UFuWm3ouuE/bbrt5l173WRm7XdsVOIEk0Z8hnr1QjYFYtkTaGH97ml6I8r5ZAyWYcfehwCJvVh2VeKq7jcUCmbdNV3uWq5mv4/bbOqd5HUbV1g29a0clqv6y2qij+lm9hMrQgaLai0SPxct/Vb0SOM/WcZVMNIPHtOxzfk5OTdz++Pn/7IZpHd0STGGp32e9fbOMZ/J7kk90zFC2TT9PJ71+8i5lw8ZLFql226vb5pC13zyyRK6vcn3x4+e6nt+fZ+5cfz6Gl59+cnp6evH/5w+sf32Ufzs9fw7tvnp+8/f6H85fvs1fZ/3r59udzRCk5TU+n4+g0hX+m6Sn8Ay9GJ38+f/k6e/XjD2++/xOVIpzu4rIBPKZicjaO4lVT7+p9By+g8v34SIkzt8SZmHxzGEagBMIYAT1XYh2VMHTZJYwXziIQGMkomvzBEh4zgoUiB9BHUZOAOIUZm2Wj9HPRbbIq34oknp5mek7oxQskRjyi+sU6AolLYFJxA/OzTUYMGf8DkQ3r4HsQcbCCnTdN3ST6G02Q+L34ZV80sBhtRAkLXdQum2LX4Qq4LYiPZ9EdAr9Po38TYie/tzAItHZMp8C+VwKWrSaNNWhGrd2JJXTNleopvs2QVbmzZb2k5SKJ+72Mx9Qv3VECCJj9AKtbBGspPqclzRD1+mDX1/ErlBFEMKzm9ll1VBKWB62PP7/nHiACCf5jOizxgbGAbsqB5z9cphEwLyoJHTjljy8/4GzoM4vkIlo5M1hCW+YftZimPwB3tLt8qfkIXjYASRd42VztUW35ib4kK8HdBFrPs2xVL4HPrJppvlphM1QliSeTFcsaGANAI4fpPI9b0paypdSW4qP1J5t6K2wI+PJZk39+pkDBgth+ysvDoK7zK9SzCGJT1zZOOOYH68LM3O27yapobDxUuxl/bg+3DyvxBNFtAQRqAvOi6gywFwfrkjraToDNCMQXQBA3y3K/Eqp6TvrLPM53O1GtrE5dLA6CacUV/mRUCKsgLt88BAj8haV4pemxBva1YJyRgD4A5jLvlptJW/wqgjhMD1beiHw12RSrlahgWLdBCM9ffHschtjVy014TL85PV4dhEUBkirchcPNt9fFbrKqP1c47a0xbWFhFlnX7MVhhoT1fSkmy7LYtV9c2xgnh0BIcSUh2cJIyifUzrMVqKoJCs8ZLWTjCKbWXsxQFyahZSQzrVMABlBJt9cwKxN+aOcfodFxREtYVl/TI6OAayHXq4Hjk/gz4CtQnwLc5/G+W0++i0eow25gQSqtFQBRSwk1QmcsC4yjogLeASZBOC2aL3m7LIr5m7xsheoXqprFFeiEtDhhl2cB2Uu9WxXL7qLtUKWqbhczm253GpnY2JGoKK3BYM3QcMtoKrIgzADPYkUtZp+mUs2iykoYEw6sJcKjVYBNSdLXETxLzKyoQLjFuHRSNfkWa2coRyMBHQYFr2jwld0eSLyMJZ5sUb+wCrFoy6Bb9EkV9V6PbRKQJFtp0KDSdmJF1E3lR/o2snsm5RYBZLklG+p/6VfLlKRy66jXdoV8uyux+Q4paOmqVhkcsPZ4P9kehTbECgpbKq5VhiwrhMWqt/VFml7LjLgWi3g6sVUWBVHGrIoFbZ2YS91Ldl7DZBHNrgEhlXD5mce3xMrwpObpLWlIczOLWllxTMOWXYtbNWlbAdMiB9HRzpN4DNMznsWjUUqTFDUZa0JIiz9tNznI6EQ2M0o34mZVXIm2S0YXs+nzhUT7cl/AFCGys+LaCjTgxcrGvgSRcWEe4Z/FYgF6OHbI+xYsKqcszcYlugNQHwNJU4mbLik60J1UqykPSTKC/5RSalX71+gFqqV5dZtg/ab+3I6i383tIugCwvcgg6I+1IPaa/wKZcTEyAigKenu7KwC/hURihX2gLXUlPQgAXumINB1N1vo4cWCbfCyzNCLlBUrkHBEH6SKKYBgqAMoNW8Qb2DvK5GYTllosysMzMiYvEXAkhcLYAfU8ORPbIt+3+tK2EI2VmjbhAHibx17hgx2wpaQmVuYOWVQtVNFEgv7r6PpKPona0BcawgQoXJjHCPERIDcEygSEsbOQ0WygGxpbqHWL6fJc8E0WKSswSGbjIKl7aGxCl+w4IW38aJfUZQ2QpoOB/HB4TmMDi4Uh7tEw+0C0eULYrg7tGfByPK6oKYEcRbMjHt6gWXHZqZgI4obDOPkLSglHYI3CBTtqmj/WqOco/fUt9GjqvDwhOswqY7VwLUeGTgjF6rA3sNyrcXIyGF9t5NKFrjUhjFlutFCG6AZSBm/0f54BU3iNzAXojszR+7BFm27KNdSw+BaGj4kZOUQ64blOBshFhjpwCxXXbZ7xcXkEiYpIF21iT0vSL6qD3eqrVk0NY3BcyT9+ArL+yOS9pzEpnTPE5Bou0eiQOfzhvADiUvOdxhXAA36c3lLTv7UVZyJaMqUb3CdBG0T8Cfd/dFLGvlAsWYGmrPUtSUMR62ZRWyR2d8sNWkWoa2ipPsSJuVlXZfhFRNVA7lAbvOqWCPdpc9KYxI9i2LdqRQVhliNmVNHO6jIdYReGG5ejwXrJ+v4Z3KgLvPlRqy4GZAbDqh7izElqUlRQW2iTdxWG1SSOlzMe2aD5GBaxnGsES/drYA/jfcW0mbbNUIkuuTIHZmHWjascrZZBYqiQF0Le9+g5z7xBjR6amukUvlQvZwFx8xawfOi3DdCLvoncmbSI862R89Koxeo17+stqTtoP3bLuegqQjQUGAIz06lq4CJE1uk7JpbV0KtpLGFqzpybyL3S9Ir0WXqI1mZcxIs+X5V1DTCIHrdJater2FeYg/RvaaVfvk66QlGBX3c+zJoL6j/tvlN0p9hMF5n48DMG7kAXKzb/BNxwWlPKWHEkdSyC33ZTtQYRxnqrpJuOBeS4KLdo984vLY386Ad5PS/rmrm8OBnxpcdRAk/jMIl1RjMj5J8FNLBUOGmHo1ADXcnVlhxocJArWqX7oBOkn7J6diflhML9ugxqpFqgf5ezFywi1AnmEyAEXIV/MkvW90wdAs3QsJNAaVgvu3FSe9rQFav4zueFvla8CYD8sMTtVg+WYzus/aOuHF2+nx1n37OP8X9mbFOPzegktGMHCv+s6VUYK6wwFJKRLArd8G3ZO0Sr85wNUykYG/r8hNYmAMsZbwUqH/MfH3kQCXpz7BUrgOFma0tT8MDuP3+AVzN8uDreTQN8YqUFvPQ+h5s8hKE8vVJEMrvHgZFLSRq/O6ePgUKgVHH3LXGlQvHB2He29rwUuy66Jz+oIDPW3znwn8gbJiesUAlTfIBwBndW6u4hOI6Adk/6agr+JDpNkllGevaowfvn8Uf6q02uJd6c2nX1J+KFQZ87JuIDHaegGkUe9tvH4SI7mzMnvQxe3Lf21ozvXJUnbGeYu5mk3wp9VDpQRC2EnpYlxh/iaI65DOVeqbxCWvNXNJx7rkcYTENOdxOPEsL+hLNe8CeDvkKT6yFQ1PNs6NQ8Ty8oXguC0d3brV7HvBxRIwb3Tmt3GtnjFEYjBWDZm7PbMKXuP6746tI0attmYnhmjj1wSLV7RsfFJLgboBm90eo8XMlftlT5BAOBm6tSnvjzoCQirsml9WH0CgfH0DVD3IcsyizTG3aJ7ZKeB3tofH4LmIt2UMCEPtm8bpB1nfU+5AvOqjjh0bQYFhUa1QwYCnGX3LspUZ84jio1nXK+gc6s7DndjAGub2ghETUngP8yhXVx4QiS7bvKyP4ojtC7Qmi9mRxD9TyMLqP/vzrWL7lJu8j/pvGnsYsbemy7jK5DZI8XDAdk3OR2cTyhBR5C9vHTzTqBVTbrVKM3nmDj4ZcrrKj1A5ELeEGPfWBV0P0n/LnC1TfFtpjhs3zB09fkXs1WBElkSIYVx8drs+qiuwNuv/zG+LmXdml7f4SR6JNpuPo+Rg/46buPJnCwzfpC8nEbdWml3mDJRMcszkRZRzdzGO5qX07Vxgi+Dm2cHEKw7Gsy7qZx199890/f/fyu/gR4PTmqIQ2taC9/vbFmxdvJDTZVooqXFd0pUjiH5viqsBASBOZqWe6XWvq1PJsXb8G0jgn/znWNTMqv0m7AkQ9bqFsQd2/Kdp5fBOPOQoUDaIztRkNBO+Kq02Xlfltve+S0ZduqdLQgZYG4yXV99WumE+/tRpalnVrolBkSCu5rszGcaICZUSJ20GkkTHfbNYZe3v1m4coGOTw8V1bFCuQIVsddFsZgGA3YYhdk6u9WA32oeThCspyUrXZctK9vUdipNXuVxZQoPJRVOTxaqqkqQsS2jT5ZU4yMyggXA0w201G++hAHHILmCJj3GeoP2c7YMJSMBVQQ2fAnuT3d7XVfzT6XOMCfi/SZb27TfomDU4C+I6zIImdCIS4xDhJ+iUlgVv73nHU2dSw6J7uq7KorhMZumZGleNExadiibKYAy75MYmX+1VOe+T8Gh/Tos3yT6CD55cUt8f75MvdXiK1woAPDYjMvem3CIJhpvx5HknYVN0qe/b8xB5IjPHEobyTs+Y+AhvpjkHd87DdUYtqNB3XmVARnYBPKNAzpUi1HQwdx28nshXccNsD5zdiW6MtAAxqTQE9sSVUChTtgXJGSMF1N+uCbXhlkDIZ9XBO/5rPo1SAFpGM0q5OmCBKezxiUh4PfPRCATXx0+i8wmGP/o2iJaLvcbWvREfTsRHNvkpRbVzeWzZZRKG38JIZDcQzKIMkuITaVKV/LNG0iP7GcYxz+sMLW5c3ZmvzVHG8y+J9p7QzsZ3CNJHVG3cqgxDsihyHV32/sCdkcAa7+CHzmrrLGnU50Ck81TMJtQpUKLaoZ565bttGF2g3+U7AsoyFpkdKTamUhZxf3Mb7D67BZ8bw79FxNeE0MSMMbkWrwB6Oe9/+J9XstUC6gfqIqBKD8Yj2A7V6urD65TAc+vUvJIUu6MuCNUT6bTbwfVKPFr0l5r1o91sSTWb9kqdg7iyi3j/z7Fvp7TAhB3IM9GaBNU8RFQvW2B2fsbX+W7owbTM4y2pkryYn3ogyDNo/Buoo4BeM00zi9rXVlCHFZ1ST6marNxMoUpd951Q+sYCPjTRm4wYXIbRvrCmBIVnsqJAlXcbS7Y29vQEL2nyglbE314gcAYHLy3jW0ZkC0DZ3nRX75cl/jiFr/WghZ5+X2JjKmZglrnbhvl4YOT7nP+PIkvu+scPBuRnHemYtHYoC0G6H7l1DF1vFHaMkzjvoYMeHRdrreIQh4iju3TBxv6MXfr0FibrwJ39ZCu5r0WTm1R/sW9FgzCgF6iWBaBKnr8iriFry9KmN4ih1Svl+VkuN+XHf/bh+B4tvc0syLOh5tcqL7a67zUiP80T/Q+Rh/Keffo621BqwMba0yWH1F6s0+hkq0zyJcHZF03+J9Hoh1QxqtKWDZo3Yt1DJE3ZmjbXG2yWXdTQAhYjzccQryVEpD0JFq2GrghUEqfZOz2R7EwIJbXwCzc4OkUB3oiOKe6cSwtKa5XOG4tFBeuEEilC9sSzhRie5XT3QnFwQBndgGBCrqaB2bUVeJbBWz6ejFNRfeEOn0uBv3uKMxU0qqdKOBtYmM9DAzUWV9IWtJ/Idcpra/4RHTkCnPsXhtYDO3dpDWg4gCi0vr/tdRq0MzPYlkK5CWi43++q6Rc8BWOKn7Bzhd0hzh5qBXSGudWibFzEBIf9rhp1A3UAExsHRG/qtmMVuLrvYL6NpNMedRFQ6E/3q4D60Wv1fbcTyeodRTujfNkstWpgK0MDSH5stmSMTIhR+80NtdTD6DCJTrZYoFhhZXhucZcjWX2cDA/7bDbY/0HbMsrRlCQmm/MVRd+FCLYX15V/F0t43NM47H9yRTYIwSF4wQ1xoOwWePmUKj04ebmXbzPOBNhYPuCWkSV6y/Kenq2UqT+gmepdi0Ci3IkkOLWBSdnNnpDdrm1+LjM4/JzLKWu7ZkidJne10AvnVS8NNFw43JHGLJzpj/4Rnb286sGyac9Zxfxb3D92Gt81fzWVXwvvN1Eb2WaD7cK6P2sUDMR35Deqzzfz5i9PTgdAQ3Hhv5nF5udYKtyNRPAFjHhcnth9fnhlOpN2M02ZmefDGkqcEap94bDv0sb7ML4uy6AoR+Iy7iCogkv2HnruQBn7w1IY6F6x39d2DwgmjPLZRtEc87h1n1oAGDjofhbjNl02dracakOvXmA6DASkHY5ZfiTkDAWb9VQCsVfGpQMaan46C0gxPSJ/1O+Apkf1T1H1GlUiFVjszhv3P13NUHM7G9mAG4ioYOq92ZFYOF+91VJ2TyK9gml3h5jgL3eQhPMYHHVbZEP8yKPc9cWG3h1XU8grZzuuAI/uqqfc73Ku00TFxfuozo9F/r7fXrch+3nzGuD8+ioPbaTLU3ZKwaO2gu1MCkL/05yBeSsV0Xl4gpAWrlXLJHfXAMP6qvtpvc4nMgEBBrdsOl7AtyMXp6OJ0gY6MHkRGXEHkSDx7whtO1VpDsE8WFyGXtbwYu3gfLsOYBNdlFInErdG5TMTxZ5GvkqpK+cy4HA/k0gwswaLLMtzIW4/ZPIVpvGUJp2wI88IRgJE8sK7WPCu6FVCEFSvV4A0lsaG0Et3nukFOAKQ+iF/2aAznpTvP4dNbVJt+qJttojHz5h8WomNOpoSNdr/0n87f/pz0X7/mriSyS4OtGNCaGCPb5aEpC3PiMyzekrDstphJDYNzMPDEtV7M/EBkm1Z89LBVyx05zjM8tZUoyQDPN0GRAR9u+x/wRMdN+PXtQ1Y/y8mg+YMX38aJGneZRMo5OjBr1VFHYK1XeOjNXmhZxNkc7atIvb16XpFRefVfyVWajtel2FKC/4yUQhv+wGMFOvEeaOR9e7SCaYHB/T8JintOvYHJ4WuA1oBSGbSEuXC6hmmmM3YkkhsGjWscZB+CqU2cMVhXxkpARSeVSOJ1knZ42Ma3MbYmy0C5W6thGP1vvxlpsUaSWFSCjgjqXbM/qTfoZggOjszsMLcSoRh0e0dgjT+BtIXpt+xWkOVsFard7Nfr0t+J0hjO9S+7A7/d5qHaVnOkvBx6vaNhS0NH79Hiu7cnVu/AhIbuGxLTm/TlKt/+74RaTSnIAOz2pgVhClN+XsLkYqsgW4llfjvHGGO5C47RvQ0H5IMkpdOH51UHrd++hZ+J4Ul2JJv9UMMbhmMddKUUwtPmt0YfucQwRmDna3tPjF6yv81/S+JIb5RBGUrXcap1G/6u9zumYynA6BTgzHVzpzQA1oJX4nnsdr9FiOmp5d3FcUKl2uBt3uPBefc94sGceaO2Mmj3m3nbdVb1ys3VK4t2+mvI9+ywQUrK/VWTk1aXdXVWAfm8PV7u6lXRGWezbNMv02IJzRMJV9LY9EunmIeM1tJRwN0MDMUJjyjcFZEE7JptFuTTaXo61Me2E7uk3ziN3Nf6IAmig84RjPp7SmIhiLYZ3BRzoYGqyL0Ee+IKI/KnI5j3oB1uEs8bGgKDvKDAqCEbqu3xIu95nzx6AwFnmzeYZnqOekVRtdWTtq3XHfbRwMD1t+/4PXEh0DxQwBSdSLOfjuxzgP39IpqLYEzS3360GlAQhw2j4dR4PnMEerDKMXPdDM3YGm3fR4OnPI+CIo1rrMkQhHHYVnfsdQ/cY2x1z143G2JSwgZP9ko5yweKHVwX40i/1ESAl5wvxfHOG4ltbb3Q8x/MN0/IWUIe//Q/KrFOf/uf1VrQD0LiiDNSuP2ZtiwBuYF4JA6GoWrkPKUpSK1kqIBCbXnw+GT4AIe7+ISPBXGZ3nEOjGCmL3+YG336JHx448SmOxPCInxvNI4ccpX6B2Wz4dmAW/1q00u5+FNbbeHdb4s4BhO7lCXAsIDrpTeDDNPCPFgOp+DU0b1DC98r3Heu6cJTpzC9HxAVcrZcWEw4iaaLC1+0LBw/usnnJcqxVMrHauaNqX/S9qNcHY47wYqfnDnq4NgyKGa+uaTUrr0IuZTMi5mrt1I2i/7qOpKqrGlxyLhAM3bIunjQEuUtTT1lsa8o9s74umvVwXWKiS5j1SyTO0S5Q87n/2IT+vH2srSC4zh+L8B+pKPoy03diopYDIMHqc9fW5lCyNWHSUJk4LfUjSnWO8V8rv+wdvVhi/oI2z7eKD5uDivv5SFr+P+VZfsAy/u/0eblwfnHM3l1rAbblWQZKNtydNSw/MewCD1rz7UM/3+y+uz1V+odahnWu73tNR4gbSiPVKKTID3wGOPg5s2BEwe9fURKl0IwQHuvErULcHEkF87CZRuCYrLiUN5vWze9tw9H6VzJf8feql73eu5pPPJine6wcv/5pzs5UzNyk5e8ObxF2dvEQ1GnXPYj5xSSfwZJnz76bhz9sywaTj+dyPTRMIb0LLdu5qqdlM4WWUd05vmNkafLbb6bx3+kiEbz9gZP8bSZObXzwqqBB48u84ZTDNoiNr+xjhDRv4fP+jz24A6xAoexgAm9zZtbStQxdPzNO8szfBwueKyNEp4Z/xyeiKS8Qhx4PLe22ah6Snnk5HlMN0qHMu3xqZBbDIvwbIJx5FunbmhOuwR7F2NfL/iMD//LxOc/i1DAa0q7dJe3id8+UCZvOT5ZZYnUddZF06rhkSoJE9rpcIAgujXZiAUzv7pKeu4a1fsMd0/nSY8i+Dr2vBBOvbZbBarB21At7e2wWtP0DrRmYd8IZGmilkMX5E7KqSnMVP09nhP8bmQK0OF+mCqJtStJ5LzQY9H/0iOPXQoqNfNwUez7AnAqyypPTq3eYPZpDG+prQm+zHeE89n4xIk4mMeWok2gozcqeafplytRHtgnZxDsUnE7iQ+hYUbKQ+OGyibxn51AVov/sNCtLPQOWo3IPRXllKCU8wtyFkxbTqHwcqLxZwyTr+Hg+ycoiEI4rRRbTLcC6/xz87YUV+irMi+ummKV5OVuk4Oi9vy/QDqSXd7t0fnx3yIbQ+KB5aERPyXJPJVZfjHyXXRmslEoxcHJR693xSfgP5ZgGi6diN1vq3bODbIonrttHZvB36Yv/LO+QFGwz3eJ7OmYbwWR5y3X224ep2drbJ0W0v9T/qn64x6bB+tmfop/YcGdHuEv4XivQjOwz/Rv8ZYG9HFRktc+v8togvi3ZTJUSvfoMGMlevDMLNpzs7BWOTaZO2cPS2l6LAWHVKMwx4h/4JYz09pvB7McW1G6c8LfPUNmKZ76uwyNXVi6tf4m41wXVlAVqg32DRpgsoKFL1XHkR19hR5jLmO5rexylkUpE7yq8vJtZmdlczLBKjDcIc7PW9drDitf78sySexaXqAajWReqaAf3wnBt+38dvBY22OBlqlEb0xTS9R5X0gtAWbYl531VrmLWEayU7NwEsm52WHH9Ns9mkBcO+ofX/5L5czntZeGEjdpZXA5Q7CPMpNJRye5HNPOugCqHXlbs3JvBf5Q6khX/FpsRIFlJjmop1l6hSgb6OEinBp00Qs20kohs5O9ue2c9zB+BuscoD6yN+ulipNRwFjNSxs9C4ZdCiKjHw89CpeluWfwuJAnUnySKZL3oVibhgxQ2n7DQJnEfUihPUUnT3c8sxkzXI4VH6msDJR5Bd8Phlb/FruFj9w1/K13DweitsOJynry5WDmYLXPOJyH0Nt8PFxwyORT/00wZR7eEpaW9dX0VPPzAL0nAyMfTDFoT9eB/U7+iuc8qut4MZCHzZn1Os8ZVZkRtPsTP/ew17Q6xugeAJZnsh1EpH1hHceS8mFu75Sp0q/skjSJeXAo+ted5dHfnBlqBBcmTskeIlmskgNyxcFgMdai1X1vxezwyizjGmz4Uszw9tsBCcbCugcR1mZPnlOaTd2aX54Ej7oObe4ecnDSmN+6rGeB9GIpBst5Ea+W1zxgmvN9dHwnhrtyGFqN+0dj22AeE+yknTdaclHKjcTDKx5p7IH1jq8XUJoEP7m6hH2/wShwhjewAWtZBvNe+G34SN3QstarojSLh8MKKQw6fLf/OjicWjRgDjGpmOl9k8ApQCQjWZSL4a/a0jzUDO9wHCqh4hf6Zex73r627xN4isc6v3bH/sDZxKFl/yFL/rHlnm91O0ww67a3B9Du6VPDfuMD4SO+Xj64og6upg9aSR+0ik6o1IFO+UsslUdqjWAczw6eLFWLKEmAgSUUvw0voLYAGVD+rIV0QMkiWQFFWGaEy7AgwSyjVtRFr5QUM1BMCZxwOeYVzTTDyp9klngWDXFOn3vCGYjN0cqgMqGGoKdKaJtPxWFpnYJHxkJycWEH8yz8pX1DwQv8W2/i9wMyhoWmt/4HzaxDZQbF53HRaXVYjt0iJBKCxQbmjktY95srHqenp6eOlAyt5dJtIJWdoeCiwIgE3od4/JiSdACnoLpk4euX/1J1yQJ5UF2yyj1KXRpwfwSTJ3zhkmQEiMVK6mVo/dGSJMB6Q0vMgxXAEB88RgMkOslAnIAa6AYDoBP1KA1NqCF0OSzwh8M+g8UDIZ9mchyP+yTKBsZFn2PDkGbfUzOZLoJjr2SPulLsqDYXWxyL+QAO6YeWqvIYMWU0oQfKwAepVs7SNrCaBKo1GOGW1ZWK2axWmQOpn9vo/su5V20pMCd4SZGca3CUp/S3lARa2BwjpS540DMVu8Lr0QPqVv87WInq92N+jykTQ5DcYNws7wj2AahhVSsUCNwrOYiE5/Eb7NAxXdtAs3x+B6EdUu/VzYHyirWALyJQx2F6VTWwLh9fW1DhtHBnJXsxtqSriUj0E308PGb0YMIPfSM2xllVeZUY/80oxWv5KB24/mrUFfn1kffvqVB5GWAJpgyYMyaaiTIAY6SbToSn/TbkATfIhQ7OaKWFyhpUQ2XVlZGYatvoT1JFHtvNjq1W3W0lu9EhQBZOYwulACDFihTaRoqXvIxLBX2poDdK9x9MuuD637T3b2zfDeZgbTfmvFJtZgfbsvsz3JbdsZCu6rTo98NgOKiv9gl3Mqit9nsfwihEivDHg4j1iXwMMbsrF6S5qA1eua2qeaDfkWPl3e1JWirVhVYX/lLqhFXayzd7rU0MIPvQVcIJD3o414Qbg+5HqHvR6JZJ5TS6UPeHYba8xj0t4xSkvIpd4rxz7sotyk29Fx1dYcv+ffNKbhUxsprFndwyq/wT2DfZJabGpds7GYb7+jAcuXcRCkG1zPZx32dvv/Q35s0XLwoE45EmtHUM6sekXvNvuVHCAUdWXRPv0FMG9YXwSpQq1NNddeVFUT2wZ7aw/K16plZrsvcf2zV3qT/SQzdm9GRw82/spQuFJh9LcQIJayXOEUyJFEbIDtM6GXKcfjE2bHndAhYmCL2HB+V/JWS61k4FNjivvRC9dnC087JUF8awGLpzAN0/dqgRvx76tvEv+XB8yEAinDPFsdKel6K3nynLWKV0fUUvtsa7RlvJYGUWyUcniZ5hNWUOhcc6ttgAXbGDTBEPeHTQzRv+0ruXG90QfHuUXGiNpo1vVfr6/le+/ca5Dd2ajj3vht+GpZmFrvbol/QuRLdblguNVUe+cXNhYWghhsgm6sgchZpxKhbrrBqfbnPyqFHUGkczqQRIDEPFLHHmIeeiVI6awLIypnE0dl4iIqtVvQalREXW0f1zwDm7upFBEvY5hOMBcI+80EqyqJ8KuXfiAQeclBFD8a+iNyjP6a7dd+fvP0af2ugVAIOZTI/v8aajP8Kc2WDksTXbY/v3V19F/xOISPbgQJHXtQq3tcEDiTAutOKjCW3EKonAI8MdahzdJq+kFVMyfrF1m71fuY7yqN2iyHqVAwN3xZLuOZmoPIZLto5w+S3F9n8c6M070W3q1UCBdTyRF1Hpm8DuI74QDFPx34Vui7oH++7A3S0h4IFbruQFLvXaLaJv101P1/eRurWJ0xKDAWpaDLbjY8oLujouMovOeLtjHE3tPWvs6ZRvGmYFP9AXaIIvLO73HXd49lUrgB8IBsZZOtcVu2Bo6LFJl3m2OSrOxFZuec6uNeFoj/azEDteyqZnkVTJCRjq3XjIbgz/TrGD1KtTrwtOQK8EJh1LWO/ZaXpGFUs7ijeaisnZszMx+WaIzT6oI6/EOrMojr6OMLg6xfu7zT1hw0z6I4ZSQad+ZD2M5vB7uXgN1PpbRPdlRH+LXuncnvQAZrXyB6lnFa+snrt6NzmDh4803FZpfmEV5xeyvNP8ZDJR/58N/yOrmGSARmeSso6CD6WrUzrDZ07IBpUlPYGudPFSosoVE4j91tbI/Yyo3koIxRUvyLWMtV37sInrD5LuCsaU7p+5cJfPhZ+fhSxlp7y7ii564TbtYMrsNQy3fREBPBFF6BeicfFEjeKTxSw9W+OHOABGFlYuPlNYfnDzcB6BxVZ0oGX1pd/MITgDbfcuZzPkkqlvLmIcUJhGNG/AQKb9STpCpHaMv4z1TNbc7CFcyK7NsVbjDjOhLu0aXMMsyOqNs+jbURcussBuxivsnPlkjnQVZJcX3SfUlFBJoeTqrro2ehQLTxwWlgwso7KXjti6w3Z4+P/j3/EakBU9PIITKD0kqE6r6M+OwRX9eNmK5hOrGX8fg7Cx0Luz76QXDXPIVXRkNOyYyl5LLvBXDwE8cq3aw8h7Dag9HpW7J7x744VRf0lXecsEWYDVe1LV3bOb4bCjQ5s3i2Beni+ZE256qoNoGqzcXZrjbYeaczj9CD3WMAm+sudbIH92HHiHs1RrM0q/kufy+GjQHW1c28w/uj8KiJfmVzoxCgNZPhiAexIrcdQzk7pBwbV4exD0uS10Sa4x3xkhlHdeKpMZrWuSNVkgPRh24NwkgFMMNATMH6GFv1m8xVgrpWW0FzE+Z79/saVNV16E+nqHvTKwU5g1cRuQfPVl8MKM6jIpCun3gi+l+7gRUtv1+uv3fx3bRsO2qPYt2xOE0YQuS5HMJg0GPZpgpPj3Yj99eqd6PiFKzr7GcXj6ND2CRvxxA+CW/ladWjNaWEgawdfSfa6bbnPLxi8mxwHMSlTU8VKDUngoxZR8AQzmZb5vBVk5z1/YRiZYg0Dzy32HV/2RtQWv0+h7ahQeoIIym33I3X6F1yeS0UnmJKXqwUOkxpDji3HKHDm+ttZJWP1rh8QecBVWhfURLNlyTI0HkFK4HudomWO0DOcc0hnDrFOARNy2a2qgoEVZDyfbP59GL0GWNVcSnj1mV/kOaSfwKnUwVdEer8EWg/aRtGPaevUg67LQNGMpvSGYluQSt2eLLV7OjncS/VEOJZZSn5lOFRCy8UHvd3hnOm4JIoPhBclNgV4BYAUYgfxTjXuZS8zJg/liNsUSF5IrIXEFvsAb3pZ0EtYHna/yHWFbNFHA20LOlXYnluhbiXabJm/xXuQjg4eKVrEtFBxUt36AwQQVXOyO1Z2QuyrkQiH6MDd3dR2txWe+2Da6bPAWRbxapeRdhT5MmJlNd6sdJ1fQRBvBCKz3Jc0exA7phBTKdf45bnaFvF3vkOAByD/x5gH6s7Fa3RYUNwsoVkAxvrhRTSRsvGCabPNb5d3A3RC8mbsN4i0oexpOG+BbvMQKaIisia6w1Yq5HuN6ULsGDMtb2Z5YqSOul3ipW28i4H1i211HFwfKm4o0o65hkkzYi23jtLD3RTAIm52gSLwk/ksl3Rok5Udycw4TYcT7bj3BC5TlwXdoJb8SSqdJ5E1r+swvX4SCzlTpRl1uYLTU3bamdPQs4nXNXpNeOCFSvxa7uAfk4BUyBfLQSjjGlNvk+/Offnz/Md3arkO3hNwlGfyuFsjB77Qt4nhIKMcd9AZvdkz/b7F7A38Tu1NgsXzGk3aqyPc/Za/P37x9+fH8NV3EKcu6Zqy6s0d2eubHy+N3DIZBkKEL4yRQZgO6lHzMdWixQNS62hpeTIWW4QS58bPWitJqDa9JDjS2prs3u411YQS3dVXWl0n8NB6Nwmfe8BZTHPYDHQl3iOlKdR/aIbmhYI+MzgSFmbhcZz36ZvE0Ku4PZPig8kpZOe+s6OxgTrze935qvF4RzJNAmX82MGmStrstxTz+vIE+Y8IL6/6skPMY77J7cThY6aO3SsHTSrTFVQUyiZJjSyewyejxTN6SEY+cpj3nd/Sv0dnp4abxsj//MnSVgJCErUHLagyl/gPiv45dRBi/VBf34m2E0OtG/LIvMKWzutc3j36antItch+/QR0U7AjOTG8u+I19Xd4SDnOSiwnRxmZDkDZ4J5JkH3l6bS5/gJ6LSw7Vsu8YL1Z8UPFKNHwunovbRZq67noylz7F8EuBOVF+0FYiaVV+xpuJbezmbAgWtNI3+DkiekgoGSutHRKowYJK2HrpJfrlWOhqnw+8FkvKEY9uPQuVsd2cvYdtMYqu/JBb352ry4Bx0Jw0vAhaXMaJ9ehiyORUJyPQV+UhEe2B1NedFStV9q9tXWW0+W7TCmmup0PGw59iUVhJ7p4+leeFkNgadjzTw37vyYlrYv0Mt3loSGdOVknFS4aB/fI+G5OkqvfNUhh/FEVByIoozOWXxACxE3/rGjLgM5M1ZfSdV8k9ykQfNzUl0zE467chZPXHh4y6l3FTYbotGnyT4iU4oJIVvwrTMCbmNI3MQddPDD5OLCtpsSRAr4tdtqo/V5jh0V33JMRUfU16OKXSHrL7GKKqRU5C0UZ4eCjvYr7IGIeRQhVExYmKKMFXl1PXeh/v3eg7jcqmrmEN1fea+fGVbiiGu7HrvfeWO++ruCFVyfo0MvlxMEXmvgD9k57M/p41Xf0twdh4mQmk0tmLSvex72FWMz+K7mg7Fya8DlEEEkkYjpiwNoU/0EBg4GAFFZ1huXhijcmTxX1Ej2PLdFjHfg13rLCSejNmtHjceI9EGyWxHRs0JJy0J1lqB1I0uXQdrK0Dkmg0VGVO6CK9ZPKuSsoIInC6Yv5finOxeEc1Z2VjU2udxxyeuhL+am32ewXoynVu3WYtNQslXgppQweecceoQXuSsq4ihnsVPUWXyXmbGOCmQTf8K5ZF4bFl75MK6JPkdTI03d2HtlPkdcfI7u9+fH3+9kOf2a0AN11xYe4wpxEzZZLA5dpOIB7tcXCr3mkd1c3wSUR3uPWYWTcJ9z/ygBrkemd6TNKjMHkoZAKVVnOoq0cf5Zp1iRPI+3WEKDRjvT2achU4vOof4jSa0cDpp2A2iAE+daMAlR0v2bXfVUnH4zIgDMlJitfLSvX37QyT3miFIbiBBp4RaqMwuFN694AT5f1xdYMk+arXmYVnuPTTp3fDmX9YmNzR9pE8/ThcWEVq6BOPcmPLYNCLmBg8/DhwAPI4wrwz8Vti7O11fBnKao/PD4pcHDt2b9pxEz1KeE746gJTl2fL9pMnBg7MO7t+CjXjcWQloD15IBZ2ZOwXIGFV/3IcvG1uRAJFwMOxcPyJLD3cxOBNAYv6PGanq38dcUE+2flzXwSaPJwO1rYYcAoq4g2JN1WsRyd5RYYdrDrozmTpbqkVjpSUPi3yVw05cB1jNv5L9UYfJLC8Qfr+9tTRUK2eojpRXSV2N1yT9z31BLRXpy9PdF+e+DbyexU+eye7QbeAnxR4nSmqAlmGN8vGWYY+oiyLVfpwyp7/n/0HOlP7uQAA'}, '13_last_layer_finetune.py': {'sha256': '01ae3ed5bf5cb597063dad40cd0f37eef63cf214d7c03789ec4d627f19781a30', 'payload': 'H4sIAF6aK2oC/+19/XPcOI7o7/4rtNraijrTrdj5mJntu96rbMaZ27okk5fJzFY9n0sld7NtrdVSr6S243j8vz8AJEXwQ+pO5rbeD+9N7cYtiQRBEAQBEAT/+Icnu7Z5clFUT0R1E23vuqu6enYUx/GHXRWtm/qzqKK6idZFlZezMr8TTfT29MPHqGvyoiqqy6iuou5KRM+OZ61Y1tUquhDV8mqTN9fp0dFH+CI+bUVTbETVRY3YtaKl8m2+EQD1RkQXeZlXS7GKmvwyb6fR0xdQblk3K4AOjyfHx9GyLLbtUQ7AqQa0vbye3RYtQKjLVZtGf+tx3dQrMQWkyjtqRnzqRAOoA4i8bYt1AfiXIm+qNj2CSvCyU72SFQF01a7rZoPl6P3JU1Uhui26KwJqYKVIqaMjaHsTZdl61+0akWVRsdnWTRflVVV3eVfUVXt0pN81l9u8aYV+vlzqX/JPWVyku64o9dt/tHWlf2/y7kr/BjxX9UY/dUBf/ftzsV0XpZBILeuyFEtCQWP1qt5VQBT5fQsgoUn97T22QB+6uy2Ornr/sro7Ylhsy5oQ3d7hryhvo23Z6e/VbrO9w3fVVr/aArLwAsut9LtW5Bd1U+HLFuij3wJyK0Sf3q/7/tXN8sp6SCuqWlUS3faaBildAedAqbbALve9evXSLrXJqwI5Rxf4+PO7U6eE6Jpi2dMsOYrgv1d1td61APhtDl8//VC0W+CRKX3Ll8sdsOVd1gLrCvlOc3YW+rjUwLINQZNvV/lNIdrsot6VwP+8/PqEP7VFeVXvRNcJ/rart9m119rE7tq2EdumXgpgYTPAb/ILUZ5WS5gEijMklZEV23SVd7ku+QP8flPnUG5Kv1vRqQr/XG10Ifyt3poJ1VPz5a6rX4scJ8vpJ5zM0NaU3r6F9sujo6MPL9/98NPb7OfT0x+iRfT86dHbn344fZO9e/n2FJ7jzSyfbZ+gHJrdnMz+/OJtfPTzy7fv35xmH15+xBJPnx8fHwOclVhHJSCbLZu6bW/yMoNpvitFMpkTwZD9oTiyfQLzF/guyyYpTvSsAgGVxCcnpqoRbNu7eEL1i3UEU5zApOJT0Xathoz/gYgEEQWCFKfnadPUTbKOP4h/7ooGxN2VKEEwRkUbbQoai3l0j4AeFOx2K5aAmy0WUnybIWUltmW9JAGTxD6a8ZQQ6zElgNDcu7oSKNPxOS1pKPXrPbi/QrakHmM11QMHbUlgH3H5XqKOLSf4j+mpQgSoCP1TgyT/yDKNAHapFHQY2Ve/QhPhoT3668ufkQle/ZriL8UGJHUzEL8wRNHsL70gTt/BQLfbfCk0S8DLBqr3BV42lztcvt7Tl2Ql2mVTbJHqiyxb1UtgGVYzzVcrbIaqJPFsdp1fXpZihnNo1tR1B8PSKB5YfGx2YrhyPxgA5QIm2qzeddtdN1sVTTztP0Ln8l3ZLeInsqUnt3VzDez0BKZcl+Gimr1AYceq4MAt4jeKd6J6LZc1kJzw4qLkqzkuvdEyX16JVAE4DN+vQBUX44wWXeDtCsa7El/UJK7grLHlVV2AlFsksVQNgO6xaSKeBNBin11ifUR9x+gVRgWY4lSyX8FquGulmqR1iP3Egw7AyjkjJQhQhfVXLIqqm/bYvRitSzpRO4P5SCC+AoL4tCx3K6Gr56Q1LOJ8uxUVME8P5ex8FEwrLvGnRIWwCuLy/BAgigl7eqxhwjMYz47T41EwF3m3vJq1xWcRxOFktPJlk68KRALX082upLnyFX0R23p5FR7S70cr4mQEWR7G/emeji+vL0Cez8pmgHYnYjbODoadh4E8E7Pxvl8Vq5WoQApsgp34drz2qqm3IEcGGj9OxztwK4rLK5BAYpnfDYI4HueBEtWiWbsBsX0FcmoQzMme0WhFCRJNToeZ1s8GoT379kvArU+GAZ38+WBIRbUfse++++4weLAc74f2/M/jsxe0xqWYke3H5FELyqLIOlg44wNqk304VlvpFQoIVxKU3tAKUODFjWjuiAUSfJ5HwMakRRilSRpkKX6mMhJ8tU3DH6R6DYbIDjQX5xvoavLzcrfK06LN8pu8AFY0equBQEUYmCwvSwVK4o8GZLYCmyxBLW1Ouu40ggHaiTnadU43SJUFMgAV0801LN+JfGhJV5lGpOVm9TVTXcgupno1LBVJfAv0FmhKALkW8a5bz76PJ2isXQElSqZjImopoUboTFWBKVAXpAbIOITTokmdt8uiWLzOy1Zgv0g2gcGwKupXwB7KDEnUX0Uj7HyWFVXRZZnRE4BB12ZtJ0NQtN08KqFbZ6ti2Z21HRoj1d35uSkHCKEuMcfxBFO2abTZh/9RX4HyJCwCZdiYYeupbhR0TP3TLqBag+8AKm8JVKJewgTScnRi17LRgMr2iyNGlFJUQBOsRaMPsJjiLycEFEk4MhNe/1J0RSc2CgaYGcrgNvPCpuScE1J8QuMAJikHf6ZBnJteYRNQ1KLZGQE478vc5jcCrcspOpS2YF80eYfGR7tOG5GvzLhrgGcxcmrMxpa6RTSNSUQ9exrbH/PyNr9rs6cryYDmI0N1bbX/h0XE7NG5BS1gWFnfydaPf6nQbbbswE6UgCMCDF2N7qkbj7Abj84fwPpiLT/EFiwLQU0qYM4C6ApGso2Y/g7U64tuRF4lOUx5rSsxDrm3ase6Tjw3g2KXID6M52bsbQZVQztxatEilxUrrEjj179wxjCm+gQf/ppvD47AqEtQ5eomICQkM6+lZyIT2jXhzl+vAFDMe8enyxJkcg++qW/bsLjZM3HAnGr1dPCas3noDBo5MyNyTnwD7wAKte9QjhgIhDWx0IJxrl1sC0ssinRaCGyeJo7IOpDWddMu4m0Xh2ZJmHGoZxmtAC2NHnb0zH7tDnXeQVuddJ61132tFART4n6chNgQW5ILqETan4OShpJn9xHQiBAJs6yrS7tEkKsRCdmM4ek9LcVKXsZzvjZIIJL/AxDYkhGaFm/A6n2DdiparK96tT+pqvSt9MAcvqCCCZsRVFwsUdc3ZjSZAhnIHueD0vLnkVQQzdoMsmEewVRAj59nmfNJuQNTM5mkPW7OwoiAcK2FP/YHbSXBx975mJKDagusiiqxcFYQ44a0hwUUSsCvEZu6ExkKNWeSTBxloZF9wfkMLAsM6zCgfmmhOY1iVRsULNTYfM5GmM7qhQUHliwHF+YaxNXf/jrBRe3k6ZevZfGpXsjIX/vnF29h4uGmUA1Vu1tR3giNiPSWtOnQIoaMDdpovhEduiwrexTT/pPlhFUmgvySKs9bm6F1D9SnFd3mCsNqvQ6liXA2OznnBJScZbOn3XIYZVN8BOlxxJG/DFOVGhkYJO3uCg3VryhOtTf3l+q6qm8rs5UnZ9w9/kFvrlGyihW5yfuFuyc5bmIUl6ma2+hoceYec4qBNlulP0M3UDjnpc0p8IlE0DtYrxJqz2FuLADzMW/k1ymTJ37JH0/f/JL4r3+QkiZREmewBQN6ygXahE+jXiKCwZFpCqILWhi92vaoEz2opCugenIKWO4SW6s0AmyMzVyucpuBBU+M13CadgavB9h3Gxj7Nm9WQ+sAX731SvuRVlpTxl6n7VLRb1IWLRz5pTQNsbkQpIzMo4u6LvVE9paG61u05eeOVgXF74Nqh3p84CNgY4lCEvc/7KE1TZ252gc2Zr9yVkSt0mkmSB4/lqAmcnj03IIFwUz4LXRaoBiQX6WeDvzK1fSyviyMxmjGMpGVLTbzyBpQ7xTAqWr8KPhVOR2IXdBhIfHGnRxRzoc0jUHF11MYrRngKJK4YTeXjo10Bevn8gp0guV2h/+WSNiJJ5mxivKFoGwmLPmkIOQzxAyAoPXRMiAPhitjQ91/GVpsMh6C1gPf/XSHQ2/UjYyJ2mbG4u70mR450s2jGzXKsKSfZ3zwziesJuvaQE1G33Pt2MovLxtxiSWlMi17tW3qi/yiKIuuCPlrhnw0EoT9nnrZ7cC2PmOvo/BvxbFy2yuzsIApeHbOv/YeGvu1xMG8Rlagd6Q31E1HTsIukeUmTMyRZFpEGoD61X8OIZXKfZ3EenmGgM6Z1X88mbhQJPK6ugomSZQJT/UnsGDBQC/rzaaukpPJ2fE5/M8DJLHVgOjJcsmapQWoDHywvE5CHWGrMrOILGRHi0g0tI1UX/wDlNWJ3qyTnKaCQZJRDtrDeL5h5Eg9sn4UF4H5gR8oamcROePWXG7yT5ZTxheTve9dWVWJHROihmvKG5owKsVe5EoPyBJWAwEuQfBHAVM43uTLps7WJ2HwOuQlBG8a5TeiyS/FQgIBk+izAFir4qbAqJrFcbhFDJF5tqdboTAa37BRSPk6Ox8u//P1YgOa1LMhzdKGvkB+bfLqUiTDxb2OatmPOt3ui+S9DMKY8yAfGazUwDLTkIe39wqogCVxUyyFVt7k0/QopKq1rq42uOzLVYFppF3d5WVW1i1Oh+P0WAVQgejBZ3/6GRnqilpXxhpXu3ph0DXvZOAfdbCo1qLBTVmp6jMZjNKa9ptRWis62t5npmECbCrrurnSrvYZjRPV0ixAobwo6+V12Cln6xWOGtu376qrNsqHq7/BNuwXwc6NdfCATvod7UBAio5RWPn7iLaqJR+qDYMNeL7r6iVMmaFhyWjZkL9T/D3uGSRpc/KtX0iQfrbikMjkw/09x6MT8BPItUw5ucowlTmnhWltD1e4jDurF9403zdAAyZHOw+2Zxsd0EPZ1WBZA0trFcFixoCSw9Hr4xS5Cn9huIH8CchetSE0CcLx3/pmtmeMDaCvZFsvZxPdb8XODo8befjNQq1g+ETGQDKZRI/JgxesK8UmVBssEdQTPYwlP7f1ukN1RKKr6QnaFNmiNl1HpJOtVCqs7NoOBaTyiOHdUEHNdOVUP584cpdEvAbey13pR1eF+wnEbU2gKG0T9MR+EmFfT6aSiFytsGgmnfPLulrC4lvhAjyksZrNCKeCp7iaHQOmwo7orv5mAQevN3W1vrB/TkoCncXmG7k2HLjmqyQrRQ8uegVaA9EiedqDtekHH1zLEF2vKZTO1NaXakBunNgql3qplUa9t4LzK2wz8v65iHgFety9L5r9eKSf/oaEyBQZiHD44sirbRWxzA7TWVfvDJHA2kQZJWMI2Z5ihEb/xG0NJcaUfZRfYyQyqjxK1TwgrGIwpGLIwFrqjVt7H1eH2sOsJgc0211qr3brdSmk3qle6egdpX4aRVftcckQEtoRcqJLdKemUR+KoWYpJ4wBaMZNATWjYrBdmJ/ms8J7of7au2sYMisaMM3N620BK7bY1M2doyFJkolsXS2WFrnkHlMFBhS8UqrJj/oZZK4Xl2QbxP22RLZqirXSjXDzrchL7i3iHl017GBTYTjRSCkaF1pI5tpylngx3X8lqhrMKOet8Z0pXLLeh6ZeaG+ZkWsKH1VyoZ/PEM6567BTyxvzcvNmFvbzvsqmX/0artbU3SZJbLxmNmhY4B9HT9mSyOkRgGZVfvxY19Qx9Ri61f6z6ZIepYla6swn1gIs7ydidvJUc8OybotKAA9vijIHBeYuw23n5AKDD4Q1j6N83aF5aV7RYJtHf8TR9wM9kMBAryEIAIi5P2xmgAolHla7TCvcS5IV+woAwSkQAKjoYtB4gpWAHsUGUGHNOYQgN2uGIYeJ4sdyldGuPBNK+JJzvyMh4Z9zLSIPEaP7gs966X2IS4qMANow8U9HTLmrclU0KpJw2Iqn3kPBXumlhydg0lyJ5fW2hhbbGJ7XMZW8N9SKvolOVCSTBnJoLCKeYShBMq0ydbSnxwIbNh+NIymttp9lW8Bnm7y5C9VUn1KMWIy1tmS31Z//ifCcIodlf0HzGWmcUoAqcQtTs7YNbrKu49d4Os0lyTza0ZmtvuHIkDKNJ0eW+YpsDutQYmMJjF6W9W22LZbXpYxSoNDMNr+BhdH2GCCnoNJ0fy3u5rLEGfw8B40PFXISt/BM3mn8mOKJJLZ5Fg73UZQBnZTCPxHHNrHIhYF7WQeqfeJFkboRDxLHeK6QtaJb9HLPg3f5ubJv2PycMF8WqjzMmZVIE1yW4CFfoVNsw9EjociRsaiRiaXz9PqIek5McNqR3gBo5ZEn+gls1UhBhP8wHXhy1EdhgpG/tQyedgvaI4mJNTRS1RU6VBOEZ4dgULmp3HTA7Sls0NqGemC+xDbUhKcwuvG405AFZ7+0Bd/UsXFdZUdLtjSkb/WYkceFhs5xuQzxTSiwhpFIexdJAVEkdyh1dMCsR0rhvL83TMRO2wl0qQ7FcIUCs9gOPFKERTvYH1SYhPOWzklLFjUetQnXADMrgsYM/hfshP6uzdkHvfDJPU/cAYdl+8iPqcEFvkxG4otkmzxEh2+aDwTnsOlLrgPVvtfsIc1NGHsYNIf4xHQ5ZoXvzesnoK98O0+frh/ePrmXyLFXBpk2PmJRLr1TSobwvMLzlqdVB9xx9wZ+GsRoKmb9aZkF8Yvz0j4Fp6yIy6bebblXfm14bTjwxa0fdFXd+wFpVA8XDY+lOOl9N2ZcNrTSkBSRwRpZ2QTKIedCydg51ejIlAdHduztjrOQOt1gm+jD3WBdMOW9TvQdYNvufGllHoNtB1r/Z5roctGkN+nLVb75ezI4UgaYPKSV0SEtyS/8DV8K22Vesnbo3Eu+2aY/wpT7mb4l2odu1MEWFCTcq3LxK5us/5S+IvvlZVWJHKOg33xgXiHdP4PxRxDinxbKD0gYywN+xkQ+QKJvRN7uGpTpRleIOjwTwnzYSsRLCybr5IkRe0+vXwKmLOyHlt2zGCtwN1U/jfnBV3vjxfP/WQqJFAa9xhnSk4nZiR7ptot1CEmj3kHhkyMZ+YTnsO7Cdo0RAxcC1aO8ulZhYOYlyX7vrW7kWDdMwv+419dt1L9ELTdVe0aSyrUNEpWibX8mf8HXRj00WTA0aspaOIudItyh3TOkFynDAfSleFU5gUbrySJ2JT1LxuupUryqGmT0qbCi6i3baOSjTNE2Fuj+Kwdt8YBXWhHNLq15A6elV4M+Wt1WvOOWpvdOwWYYuIaLs94x8exAhyE5ATrFboNignJpyGZip+o9w+HhyT0TSQ8sklpNXypL+2gYsp1uRbPOliqGZ9I70WRDeH6AQg9YA5bIQyR5FBKtQ348bIh7KVYDtaWEytcZmBvC2YmV7hRn1998sPf/zXs7eObs3Pls9ofZJ9LZO7Gdmi18odw/TuQHZjYJxIL0QpeMh9DxENEuFwPjrOh9L4cwDgSDCLCv3dNngWMlNFCLE2aV/P8QhP9XQhBCkQIHRR8Ewgn63WoZTmBFDjgU9TfBm/w2+7LtbLYB3ld+IgWNTsKQ8SQMNs/LZY3+0Cb4hLRzCk63m2mvMH1NhvMcWvKpSR/+NNIumiLHfuyFFB34EbfUXVEwtu2NAf4GKX8cVN92Ff3Ikl5+Tga248E8kxmbaL8RO5GhkzsLM/3ZcHiEVtgHS3yJrRyI/hiznd3/zsPT8iQ9PiTURPMHEHiMfprU2xWKfP/7ly1dzvLVb8lo9j4kUIOvcmPhGt6qp6MiBqa18b2oUNIviNOwllDdzkC4RkCHpEEwEEGiFCs1swKGTcC4sdZaqOCutAEbJyTd2ZHU+tbyUZH5K9W2uVyW/YObalTjOR/iPjSEDdskWHc89tTPJZdYRJ/64z0ZPWl6o4JX4jmj95kMbjkPlCXJwXDklewohjMTZjwIiAX4jgHqi4UASef1AEpO2MQenFTpMFIeqEGsmPsHYCQjwoLknPJ6YBw8OmDOQ+Fwg/6uYJCZpQ6HRt3y7nwJjrOTEJIT1/3DDDztp4KpxOaVtOfsZumUcmAYHPI6xYYGtC9mM5pTatYXC3H8zJniln/XGKbsZCw9/8V8sxdtbsviH/+jtlqDboCJX16blvTXVfaZf2PglB+V+WbxVeanbZbMPcOTuoDCanGPRH5ki7dH5/P02fph6lWDsVDHWkxVf7RHq78+8erp8R+thyzAauKjKh8HWEDFT4DptdelPLJiGMnPPDrzgfEPVDW+nDmbs3zzI1BJOXLmRvvZU1z7b+bWaj1aSXty5loQBMoY783czI2hcpo05mGopCa3eQh1inw1czkHHNe7ozU4XrwAL2xXKUZ2vUatNVH9xf2vbNneOJOJuUO1hIRCsbelKT6N55OhqfuXhVwZdC4451y7P5dH3Ukib8o7gFxvKbkvhb7Ys3zw7D0JpEbk10dHXDxKcXZY9tCwkFkV0vlKE0L7vMc8paZdnkmLpinjHGsc6L0DDh3USkmVggt9beUXuthdb88eH7tEGEXPVzTmqLsHNdXRnPuKjv1P7x0onsGBpiNNTX6XgeHH0wCwfQ4Wns2iJUxfvO/jSXdt1ei92XGRGcUx/90usAGDy0UXgYJ4WWFSudRdIyj2Ec/nhaIhw/vhbmDuwHa2HQKCu1PoTAmH2YXpx8O0B0joF1ExLmW+bYdcxNFMu5B5yJQdN0/JZufBKIn94lsZSajT9QaWxti2Grw6MtmoW8nR6r1alimnK7m6Yq9nSpNhP0a9YuJUGMbGttOGMaEx1OTh4zlMHSX9JHF4lRHaYDFNGl7Fx4cI5ytw2WW+9QyP0aHhtputpY+RMFiLH4JoBHrvbgTPL6w3mXG+ApL01+19Pz8yNfPwWHRvtMt38qi0dXQ2XLftVm5VeGXXVBMv2xTVrqNTG3oqPom+PebeLqZPspgNplRaB17x+IpdDF/xox8m1C90dmRM8DoZtSxO8Q5KmDMsPhfaBd3jNFb5wSMasRKBTLLNDxGLMSlCdrU9ovJBZ/dEveMznnXfQmEYKy/e8vFjSV0pXE02Th7yONWCdBLaY093VVlU14lKE28CXlXQYqmWcRatoPT+qdHmqezlMlUXQiRcd6JYCrHZwpJCmb4T+1g5C9lUv6Ze5OWDCYGmyZlzTSDRas/uX56SQZ8gBgT25134H0uwQM1pP4js578+lYKOUtfJEQiJyXQ4C4I+vFHWtoDagHAogC8TFm7iD4cXuv/loydPhKmAcy9/gslOe2SlfxvNlKBulNht8u2RFS2DzdsnwfqjTEHuNHWnR0Mn/NVxO6YrScGhGssGW1BnDKwTaUogqDBWCuE0V3DgcZECVgp13j/tb9FIeI/4qS0ryZGkEMpzKafWxSWecKBN323Zpe3uArmgTZ5Oo2dT/ExnkZKT7/C+nQnvG4aCWY68+HWvMMf7cqFZnimoKa/okWYnywymwwViRlmVxHAaJSwOtJ0M7MwnCVPr46nFBTAlEtOXKR8z1VOeS7DDq2qaFZBj5fkv1cb5TP3gk3saXQuxXRWb1tkJmtiHR1RNXPvDFelIyffM8dDU/xB9NIOND3T61UvoLfybVDSRwQjH4xFPiX0ywzW8W24QfSBlXzf7+d1pYC8Z78kJbyXazYc37MCSAMH9CSyXxYvpwBH1olvE22VoH1vFQTRVn6g0xo3xgZIy6ba09BcsejuwYzhGqtFoiwDhfnn78n0AIxRPKX77euJVWYU66wXmVx2gHuiNYD+2mFr96f8dqkzYiojqOe5SrQvM9xybOMeYThrjvtsiOlaSgeZk7KyP7dk6vrcAPWTm3iM6mxrYQHNvRrJQn2p5OwngPNyqfSvTUMuhu5sObx0FHihnu00F8grav6pXmPUAT0k5Uo+JBCd2Y91IaW15K72xvQ/yRoypk2V7Z/NpdDyw6x7fWcVOhorRLSLzgSPL/l6S72xsQaMCLRbGscG1KhD8Aj1cUI99wJ8W0B3/9d0C0PdfX+1AmMhrT/y4gMX3gTCDvNxe5TDPvg9MxfzTApfaM1q95Iieh0K6LkHpW/TzAF1RsjCl5h6No3Khy5i7oitFAryLqxymBZc89BAfUvlTV+Bp+LPzQwrfhQsr914y1KVQnBERAaO0vIYuoSH5OQmmD1FVR4PBTBMpHrK60YYNtQYr79PzVDVxcVF/wlCOvFpe1Q0oQukxqEYnsBqX9XIR77Zbujdw3SlqYtLTdreVNDeq0U+7blav6dIJef0PcyaqE6xIFKkAsbsHtQ/S0YFQVevIUQHFMHGpeY1mJyCRSOtxtS0WJ99Bj6gfBXRCtIuYqsamzrKsW8e2UyKPWwbLXXMjdKoGc9DTSfk3qFGeAFm5Rgnq5fP0+4lR6pj/vw8wfcF4Q+2Z6Ohye94brKzXzqFS59voAVMGgu/WhAS0Ea50QhF3fjiyE9tuwDWCGo6GG5aMeJ76Ao4a6wOIp6FvzH3pFKD2FyAK6MdDFDrRpiXYdy9Cy/fvQmxosx1vhmm7uxKE7WwWj6OMmx1fjPDJ76LkUJzBv7Dh/UEOYzQbRkwPn1kXYjypBeZ0XRYryQ8gNfL2CqwcEwITs9onVu2XCr0DIeA8z2l+I6w5I5ZcbWiQk/iUSMNpmV42BVhGqmNP/d7cqap0CG0m5Ck02tcPIK8La+yDRYpNAoITBb7dmloZ1jVIFBRlYBqD8lbqi7esFSAmYc/vyGUSPcrphkJ5da28j+hg8X7kCz6UVPAJc7fE8jddlYdWUR92L2V4usUbq8yGnVoknAVGrwp8CdBXk/6ujJcHeXrcTLJ0FSptdtm3oyaDTpmx7JiuK4elU8RwSu1esfbb1Mrmrmv9ivb9NPpOLWfh+2ATdZ9rtJLPSvtd9M4cW1SQush2WsFUXMR/pVh881bqZ1mjrhNePHthJXmpm4u84dEEmpX5HFb3EYaUjjkxp9RbaBNUXrr1dVrIEGutxJIyc8qj/Sw5Rd4LF3YDhL5KzHp5mW/7Z+KctlN5e4q1AwwjJ47T51ILdT8Z6NyDBdCjf8daz46D+a8D9v77vMEdXmu6g0oKPLkp5JWuBd56jU51UPs6dGAAdZ8f/yny4/pixFRHQ7RXoEXnFSUcEBshBe6MBcwCsmmE12tXUCQEjWLSARd52QJRAUrr/Eb43IityPHyWq8Pg1EgQUr08jUiQQc9hQ6CLrrrqMkQ6ohY2xVl6WAelxjlCh1rEDOsrYIXckZUgryrwLK+KMW/Ad4NbtOvVi4sYI4apjqa1bhOlC3G721QCTf3i2u1fFetGyE+y1sQGuHt9fv89e+LABeF6fOLAY5donVC3Qyqx/tC9/fZt3/SV5lr8KnbMRx2lR+IBlSPJtAKXkki59AphLfBGL2LugMRUIGG3P6bC4wxR7UDc6XB62DrprgkLH1KrYRAY8hjlIHczYSsGsRiA2N4g1dalShPyjsaSRjHO4HXxpfA8sA5afS3Dvkj5nuRmGSuBFmL98WLZb5rRYQBmT160fIKxXrbx05HF3ew2O4a5avg3Y4D80F1bpNf61GSmRWIXfpt1TS20mDdYlRMBoSvGyXQ3E2M8WQ6MLx0M8qB6YBqTHhcln469j7sjNTKueX+0WvxrqXtG3ePwFuBNdPh9r1lfqlz7+qrrTJHM+czKrb6Iw9HsqfQQndJ7QEHNvNpJ0WiIj1upp9n46EH53pvXu2SysUH4ATWIR+36XC/g6U0nnz5Jd0ddxuPwh7j+I/SSfCa9EbK3hHR5spHuZB8IMaKj74qUlz6VVULb/EYGswxuQEzeyXlodcAD0Zgykf8xz9G/wv0EXn/bbCI07EfapyJmGgI+0GXJiMej1ol+9il8EoOwipFzn1XOimREaF3rt8y0tdW3wh587KSlyg69UDRrcwOrBViBesxisQS0xYtYZVBQQJw1kXXAa7/8TuobTf29ysQ6D2nU9OIaoEXW0IHSjzpNssxp4+5f6es6y0uc6vdUrjI315BDxEC1geZROPaX2AF4lIS4T8OGk2V0hDExyr6GVPVDAzrOp5FH/BC6nl0/2gaPUr/AcI00WJrwo+qQsnXKJ1DKweuF457FYujdO4vGp9GT59H1//5WaqesoqGEGhGns+PnvXEAxOOnydC+X4idVmzRoTxOMX0UKwU0HKHSgyoHKBkCCQ7gqGFVnzKl7hu1XQtM4fyruaLEelBrTaTHbwouHJ4cD726VpweNSeKSqa7WGTb6bqzCWPvHr3Tt9O2N+zpXRiew620cns5OR3iRvTtL5PHiNhbF4NMqjXhZ4I84CwOHka6fUbj6A1yG4sVcnv6wFrub/tfbSViT1fTPKlefTdt9/j6nrv5Fl6wJcvphFeUzW1cvWolEuqhnp6GGGW/7zDXVUT3DVQ8rfo17wpiKl+k5eARb9Z32ezGf5//ps1+3+LPMeG3oKV95H+ZmcacnPjeNR/ZKj/qD/I9BC5rRoaeu3d+8lr/Pp/p/C+iHLI9HV4Yhm/CkVfRH2aor6Wk77Ir/hXShiAvoG+jskx5hf/UR0sjqyDxbpm8NixD+StjCaIVBoGXVtlffCKn2LY/6wP+9dnCfp6+oVf8wNtGFMaPSzNtoztsi5bvkWr4wNp/SM8iTcDNgXGVgPwj/pIkFwx9Qs61xO9PrFYLf6ojx7pwr8yEWuB8E3PAcYP/eNQQ8limhPfgLi/3GEgNkaKsGCW8JQIqo908uiQCuuTw4oylfyAClxJ18Xvuf6vXjpUuH/0xnciqesdHx0w4x9pbVSpbco6f+RhewCxtP3wiMo+Oj975JwDcysw8+ERixpnx86U2bC3dujImlP3XlsFPiFdLcsEVcmc6cYnZvqIH7CLzhG5YUgythGm/ewZA7iHgPYFOeE2XtUNKDud1o+MqudAb+qdvtrJWH2Po6cvJg9Pnr5wgZIFLzPdOB6Q+RhjeManwzffOB1wZRUdS9orq6gUSHs6IkInppR4MWcZpRiynrnIwpr8G71g4k0vWnSqBISj3jrWJ0A8uTU/5B9VyQS8ZlN9ObDhZkxH0NBpErYRS+ZrMKceSQE8eUYnKRHMI5njiJi+/2COd8Bney71/HE7NA39+cfrDJ4XHW6lPwTxNVXGpAqvsu+sAdZ/vq/J4MGDvqbrcZPD5KZ8sFN7xM7OIbL9B9tfqGYfmhzvwXgpUFX8AWzUC4rNn46Cw9n7FnCMdPeVN46YjOmOkhKtf5LYEqz7adiL2efrhyFU1KQxp6WiC9HdorTaDp7/8g442+fB9uA9NnZ7MGZH0moWQmJC2QJNK5fe2SPTHxYd58vtvq0aFu/e5fDlzcnw2X0teRz3MirrWzxt6oo1vG0anf6aPTTvyBB+tL4cT4jj6N6QR3m9a6BoI1cPrMRGWR61iz5eWU5lgiXzMIKlLk8AkSueHE7AQOgjwe3lq2KJgvNSkIcohcGqbxB/B5QpEt1iBGCkEtvoU5f4YVcRcyNOuxbAdpS2WfmeHXgmUMjamNlD5p+ICis9cWkb4ArICi9/fP8L0hx62aqh6NXvdhpd4o6L8HxmQOe2gNmIE6iribJMnWY+HkJXj2RTtNe4jeBRe1ODvqw97O0GNA/j/WpFt7d76LtS7ts9RUPJsmTF3599w99sI97Suxyt2ngJabkp6E543kKgI5JyfAW27rA626htdpX2Cq5wW4Noh7FI6MyRm4B4sLvwRo+A8Q0lMPKuNnlzLR1cZiBn2qXHpmcAlHITyd0mZfxIRmgl5wMftTX2DKbuLRWNLho6UxwCh/rBTKI2a+9Aumyifyo3M1r7V+TdBVFetHUp5w/5YtZrVD0D8IDh2DaedhW5e5mGTc6tXMTdVSr3c2hbIP7vKpbeTlpfJyp0l+ei7+9CWV7nl0KdFRsInMMfc7UjtLzCxU2Ft9lxHZQw2EqeqHaX0s/FNvbqDx4M6xPIQTVMyp/+72L7uuChihwMUytv2Szq6/7tffbD6es3Lz/qQHG6MUBBsC9AxLRWuOoPRe1hJz+cvv/pw8d0s3JmrEsKeb5zrIwOgxnOd7gmjNKizbArSSAEVXVDjv3AxXkAYjr4Je31la5mg4+3yGVbWOI+hZJnTBzXI0MUr7kI4Emhk9AHdlRNtn5Z1hdJ/DieTMIBsACahnGEBl9Aix6dohzIlshLfBVtghvKnF/7W6fwRnp79xLluDyR34oMHxIr9QA77gjkyG9y0A0vbJoE0gfEp9Idn0f/lV9ewo/3J8fHqLd8fE6LqpSHIKV19gq9oMhzXhmsshntX/1hEb0Yb4nWkVWRX1awUIOCgEv0slOxYxLcE9r6cJtqxSUu5JncTGmxKR7FEmrrv4TYmv0Xte9CkXvROi+ayBwMdBsz/k1s58S9vST++8sP7/727se5SqZKBU8o5CPHyx2oEdMwudO00Et1To6RmztUam88H4DRTVdig8csKEby9go4F2MI44lzXQ1wBQrhhDrAWREESF32oeFKpATvrdHi5sgccAkX1LLLhXjoJTYM9iFV+n37TLG/H2/g3UW2iNsceTJb5k0FU3TJhOg1cXmGJbOmrjuZl919a99KhrBaWbB/dA9O9hNhEZodprT4BMYGLn19Uf6GXZMmeV6CIGgLPhfYe6+OnieL0OSxsm9g6FZWrOTdKqCJg3oruRweMUgRR5d0oKQfgYm5+1Dzh6G/YUIqP8yJOq4dEYjZG42RVYjC9Rja6MuBZv/68udTfeI7UwPoXnxoEHJH2MVoYgWt9OCv6ho60l8F7sJHVJxoigCHjHPEABNM+sgXdffNxa5AexxfmF1q6yotWQ6MQ5xRGdgflEqDoaursfhKPY4OPkGucb8GmZDlmSfgvDeAndLL1Ud272CPm5ntblYBLpbw3ipVQ42KvL/KANpTW640qg49TKzLDX6haDx7CZH8OY/ue7LpE0i6Ft6piAEFGIGr+zZ56OV+h9cimCPb7DS7cW6h5nMmT2ydW3qnhsei6PndqFMvFkleCzrSiC729Q3JHUS99R485211ehK41w2qWWBSfjLTq8vu+olUUmKrsvqWKZKTE0udBLKTI7AUPvTbPocoOYIbAAZO0G/s3o/npU1zsgXxb/7b8F1RPu387/3QBxKah65J4vN2OnD3ti0gbB1f0bmu15l7TT1etrUrWRasxJoWU44Q05UxpD6vXN4zF1Zz/vOSohzS7Hfffv+Vzbm5VExrWVlci8RDZ6rAG2meqeWLcaK6n1nvWuicB0ykqgw2+g5blZ/knCXjkndxLVRZlspGX/gN8omdBnZH60yVP2cgnFQ0bi4u08lQZT9bjQHgUjFU38taY6qbvFyaHjqHy7k9XZHSh96bpPJ6KUhn8vl8X47Gvji/aWMoRaZOOdVXGj7k5ibN6qsMHT/z8lqZRvqXw4i5aZEdDCkh10j2YTdN2QioQCjsQM8HceqzlR0CZRgfk8Rsb890UrJwv/qUZQfBGSGQyWT2BZBYuucBSEPJgw9JcKZBho/Bj1b2qoSaPyB92TgKewEchMZoSrQ9NBiu+jualhnVvqZlrHlQw35Ctp5/3E/no0l/rexr9nR3vwcB+SncDBD32/n4LXPFmhmjsPAWLSy9ibfYTdK84lcB4H2MurC3bo0VdhcpXnaP2+0nuTtIRzCkkadPy+gYaX7pgTYgeBpEvMFE/jJ9HlIOPRJMh+/O5BaEXcdOeEEpocCqbsQl2nOuaTzS5BCWjurKERnpczgLRwj/fX1W8UKYWipWSSg5wacqx1+f+UO9f3DOqLg5SnpdZOIU1CmWA5ap3nGR2ZV5RuU9Zq3qQ19fGbjqtVvZSWnTQzG7pyDeZWytBjV0YyNtRANpQncB9wX0/YzmntX+c+830WUCjhSWZ7F3pOjig/4VtdXreix0vTFfBq+qXSFOPd9DIivlGB5OqZygws8v375/c5p9ePnx1LtsQXubze2b4Tt842D4rK4V/DgN3PzR4y+fpu61mzJYVhfSz6NXRIxfGurd1rD3gk4TS65Lh27xladeZBC5Ludd6iv3/VhstC7pX8RpJ/w0QdG6hvPaqaSTTwmBC/dg9qlY7ghnOpwPyt4H11ZbRzzsmJZTn2mG4bDbQXVwrGX2eUDlHqzNPw6mrH+wbxoNJm2ZsG/2af6vXAPDHiQWwck2TkxWAmjDPcnPshJM+NlEOnA/mpkyaFhPB01m5vTnLrLBtfQremlwVXgRDk7eBffW7PHFyYAk5VwvTopIk8G0s0PLlBxSoyZV28+xlxZhVNtoFwFCWVyyGOEbb8AWXzCEi+ExVWOzGBsxFQjhndI9LCbB9vb5+wJKX5jaDhUnIZkaN8tBJreyaZt6KHSEe8zj/65eD2QW6XMNp5aPHXkmReYCkApJUo+qbvF0Yjvj5RFQjDLn9HjU0+OR671/qcI+8OgI/SLP/RGYFRnl18wyCqHKMtyez7JY5/XAvfqj/wMWo+RsbrwAAA=='}, '14_compare_matched_control.py': {'sha256': '468e5100dd649b285e8e7e4d684720b04483141a8b4c6a62376a1f6e1802fadf', 'payload': 'H4sIAF6aK2oC/708a3PcuJHf9SsQuvbM2Z2hJTt2fHM3qXJsqyp1a++Wz1dbdYqKhSExM1jxFYAjaSwrv/268SABkBzJvtxtJZsRgG40Gv1GM0/+8GwvxbM1r56x6po0h3ZXVy9Ooih6W5cNFYyUtM12LCcbUX9hFaFVTgoq20VBD0yQD+8/fSatoLzi1ZaIfSUTgD05gdUlSdPNvt0LlqaEl00tWoCu6pa2vK7kyYkdE1vYSDL79++yruxv2Hxnf3/hzYYXTKPO6qJgmUJkcb+t91XLhJ5vALDgazv3K+JRE+2hQUrN+JvqcOLs1RR1C1BJc8BfhErSFK2dr/Zlc8CxqrFDDXADBnBdrtHLq4JRUSUlawXPOtriEwL/vK2rzV4CzR8ozN6+47IBLs7VHM2yvaDZIZVZLZgey+z6tFQAenRzZtfMTk5OcrYhin0p8FHGM7L4c8fR5CMtmWxoxpYKUg0KsuoXvBHbfcmq9lc1E+dMZoI3yNZVmuZ1lqYzBzKheY7bKJA4Wiy0TCxyLqI5Eezvey5Yvvos9uw4GK9Yu69Y/m2QagL/ART1vm32rYbvxoEXdF+0q+jZFd1uC/bsphZXcNvPSiba1AhyminB5iBlBvSROwJbSv4FiDanFmzDBKsy5lAA0sVWm6Km7ZCq0+TFK3dDATwQld3XvUNzrYAmT1EbYpTmpRJidb8gtUsXBa5JcLVUKxPBALBlt20M5NU5cGAV7dvN4nU0s7gRJM1Boh3cc3JNiz1bIn61z8e66iQH0KI5qNqkvAKux/oPqa5sTtgtl21aX3k3CCA3grdMk9LxQ1GLW8tY7TcnvMoB1+o54Kkk2gsqM85X57SQbNYzMjyMZabDLTBAIbNynrUXshVzPNalPo7RzhQOAtqAAOQZicxoZFirZRLme9JdOACor5mgRZEKJuGCZfJ7L1Mjqzd1kXdLM3k9vZLdNkxwlMD0GiSVrgv2EPK63qQNkMu1TUyq5osn3SWXEu3eiiAvYriVckY2tSD4Cy6gPy/fELDRajxR1woCealwwIxBs+zIAMsvGfkElhfofS9ELXpuKcWBKaKVlXAwhxVqX8FalpAPBheJyE8ErECU/F7zKjZbzDo0+tcNh0uqGiXm8bdzQZnYoqhv0oZnYKO13GoGof2W9Jrl/bGoEPQggVt3V+yw1LMX8PMyyermEGvWwd/IOTWZoGuS965W3vXWAyUsWipB68mx4gMTvaY/RsIclVBCJRF1rrUe5Cp+SO5c+E6+PCq8K3y0cPY35mygOQnY9Q9nJhd808KEv9skK1i5Zjmqf6oADSs8WBDQx0F1gu2BMzA35O4+PMa9MTDgUgTbUrRn4KivpKa7EfWarnnBW87kEiUUQgI8qgYu6BrQDsc1Cn9cmat2D+px4QyT8d/GkmnVSj0qUMkv3VlNxGBY09APo0yrMSXVELawPJasjfW62axXj5LKK4CzCMyvbnqMqIQ2Davy2Bu8QESXECzRKqZwJavT2SzEoom34CbIi/Wohp8lZQ3eByxLCWJzNrs4vYT/DBBpai0i9ZfniHs5BC7LFqbjsYM4wg3rqFS3EXvEHl2iyZiTXMUK9fp3CGNnvitjtzSDeCWrWElFvF7CfbRzkqn/VUKiIgx9HTnHYDCnVQsXsgZLmllb7c6syKljs/V5z5JTNSRLsDAqLCzB/K5hI82WlvICBuW+dOwtOnVg9DrusWv/fdvzW7kVHFJ+hVZbFts9fiJnM2N0n5H4+Y8/9mi8u0BKgL45eU5+VIRY1kDEwHPUwYZy42p0OLYMHL2JlW2kOZwOIhyNJe0Mv/77wtovrSEKlbPGYg+WOc4hjrRIoH/TN4+/MsigJNhcR6WM20V5QVQpuGNaxB5VygHNPSLUkIPlMS5ZURh9MGkdpmwkr9XmcocJX7tj4NNKpnzffRJ5wPqWYEJ6gVEEyVEq6JZKx9+aA6fgL9ScOyXZFl2InlTrRmZTySAFyj2ca6Q7lRCIu6NbQXOOEJhGlftCpZjuAtbU2c5DBM6YB9G7vhe+4UBT4SYW0Y7nEKKCUykjz4PVDWi1O3TD+HbXpjnL6MEdV1KQyrKu2x14I3cKNCSvSzgry72QDa4ZZQDtc7cYGO+pmZEyvA9Xkqzw9t79MtmCJYd1vZr+wRPgyaVd7Gfo+YbY7zNIkpIvFCsUsK6WUIsw7DPYw7DvyGGiss5ZNMNzRHpNdJQ2RY6pYkB0AGa3FgcMSpEyamYWiBSJTqKegGkuuTRgVSRVVZHoG3mEGyzUDmOEOdUWRZfDIW0T+4QW5Y5jwSD2bJopLSyJk5kaKzI2ZcC2tBmBcEeVCQWTajwRK1qKUYGHmSxCMjr/pNb/GVxTcvpa1ZW6Hci/rxwqQE5gycuBAws4+V/VRjD2BbMcza2z52THiga4ugdvJJRhsxLYlazAt2csh6QzIb6li97fNooogAKPQyESUkQK1jCKpR/RctBOe3eIa81AsFmIZ9/TRSEURcsEcIpE6d2mxxfgwAIZ812npqKEM25ZhdkD/6Ks4QgTAkIdVvwHY42u7TllP9YzBMz9GpDVG1ILvuV4oAAZiDH4dSBreMbBQZQSWKJyvjF1FVSBtq51gAIbwQkgLAPtg1EISxpVWcuBUqAkclKi/kYSMnmJEFRUatgcEBW5wEwQxxxskgI1ZA2QBeA1Z+kKb0UNTmpfQph20Mcxf6iE7B1seC7Ak2r96QpIaVc58vTLKcQMghO+nRN6q0L7pmgTuV/j1jI+gwhpjtPoEFfxGfzxx+S1CaKzuqiFiu+jJ398+6fXb16juX3y7tXL85fnJlRBpBAsJ2vqGCZziAsn14su58Np5bc7vXaXqK1XmgDXp9nt6O0O2dlvOcKcEFu0LvwwATHI9gDJfLRYuOPoa1eb6D0VBUcptbjtXetkmMR3Y1eSnP5wP4seRfRp8vx0QCWEIodxIpcDGqO3O4qCHj8//WF8TxDY9FDwMoZY+GxsCvGAa8ObWBTsGmXY3sdwecvbgsVd7GeqC44DCWAKtsVcaVNDpIYC9hoEv5s/+36p4VUKgTza9kcLzdk0L84GvODVwiQKaPh0Tts5JNx1CG1Y8wuwZMPbFu1pvxBDLarSGQRxykS3SctBBYB7tJQ6hY1usaJtXjlWZy9n7uqt4LlZBkJCaNHs6ArEyJRMQbVbFT+CKYfAMu6HscgEaq6Km5A7Nnx19qfTfjoraslizyypsk9/s5BKgCUK7BIJaqUDixPam87S/OucvDRmpqkl1w8xK52+qIyvYGbLma3f5e2OoLN/8cow3xefHs3CLH4Gtq2PdhHXhQn00hZ0Kp02Pwp+oGznOkgcaGxnHD2Zm6LupyPU9SHg91H4M8AbN660BJ3dCL3Gggf0ohTfojjKuKN2OKnT0f5gF5voHCSF3PGq1fX4GWbp95ESejWAcm/ZD0ujy8vhxgO99FUSmBGoYeQtDAwTMq9XWNie0WxH1OYdmLFN3d//j6rVvcsZPk4V+YywOhXpkUKginmPLjHVguHEPyNieDknrxKrytrOgVnoyZkTdTeqltEJTWwcxHzkiBBifPCfjIFdragLt+4cG9M7H54f4EfVwNbI3CokPorCAcN30qCirG5n7o25+40sXvl2zFzAzDlAb9XHH3WDErrzfNs9C+qFpmS4Mns4OyR4YT4eeruiAZqspM0q+guoqVvdwH+0LUg7V/TiZQCJ1gSMnH5nGzuap5zq3zMrXCBQjVFaN/YAzVrUmwVqan8r+vgZ1rv+qbqoHxchH4M8IB4oxAPRuHKPYz7RIOIihWSjleN1xYdieZuHLzE9nqg16toq0pfwos5AnXqDEM6d6TkVTHoFoegJsdqm3Ru5lgT1Z/Fz35/xNnzxVpDu7ydPyG+7A/m8g5TrrVZX8hskQx8Zy7uy1AAK0zY2FWTLg8SnRYGmG0ZfvPphTm52vNAFxg0XoOORWyDrqhxuSi32VYfi7NUPCZAIImCQS7ibXFVJ9tJL2lQBvTj0pcy+sLeARE43jripvzq4sVO6ZOVg67AIerOg+5zXsDSHc9yAUOuHSY4P2DyDNBizzLmSLrBk/bZzFyGvONYOTGI+J6qmCQlujussIwUB3cM4VOrkG/h8AGmqm0YltpF/DXVVoKNswSUOcuibHYMj9L06lQQ7X8JAVzDg3nnNBdaO9U2OCM4vJov4pJ8Yp1Z+xVzc5AXkK/nsB+V2oKSZqMn5GQy8LXjjLnAo/MBopYlzF6hRJ+73kfs5AZaVvnrkLRYL/O9y+l9meV/jTzHQv1GvZVZZWyZgSMaOm1JKa5+dTvwq/FdyB8svnvYp09PLe2JH/fDx6eUyebG59zihsbirFQPTzdlDqzPg7reg5tWjV2Nq922YTTLYLQ4rRpqF7Lb1WHjhV5wC97eJFuRIBaCz2gO6RosCQNYIfmt6HeNV8movu6pZV14d7KILyoMr7sqlI7f/0wQV5zbMElg7hsTEqSffiBpbLO56h3bxdA0yNo1GQfhIDOYATTaO5j3aXuVAAfJD9hHfMEmzMK1FHoYmVaPm2gNc4X1Gv6GhVXbu+UudIfQ1RjCbFev/JtkOgzfZVTiC4mRnGdYHUOQ90MdEBspHtwwyPTCjEl0Cs6cw1PtmUr8A2Ya5a0bAxLJ/A5cDq3p19muYKr7GkABZHKBCnnCpnQh4oqxQsVPyEFfAAKv8bWh9R5d/1au/2tu2mRaM/NwLsTP6rncloQ6bhFrbUhc8NK6dgX3YuPYGdjZiZvug7XssLWa4yuggmqeXOs+dtEzTlYaHbNrxKsCDFhFBum7Q3pNbA/AgwaMW9RGk/p9ZYpTRv2LXRoNl9+BldgzARs8PyLIbfkJ4acu6AiwOhwgOYsLNvivpr1Gk/dA0kGZkPCi7rMkGLNaaZRSjSt4SHTFia1yFgRPV2q5ehM0zjorQBg8g230BUbd5fbHBnDYs9rHDhpwQk90oPKSiKNbei4ZC93dI8FqVUdhwbt9gLBO+dqlwryKQjYmDemrWbU0BNrBKsBjcIb9l+WNszEe4cvKfLWseWvtOtxKYRy/mPnmp+NMwbF8VTMpjr1H+syeE0joCRk4D1ySYSjCvMLBm4Es8ROaWA3yG1Qn5Bbl3w6VqPM0guZBs6k2rdzCDtzIki26p+yTovCatQWN34PiudKKTTFi3sEc2+ltlnsOVns3mg57XvuU7uwJvZZuKci6cMi7+6Pt2IV/sV+k+V68bWrHFpNHJF95EPWlwS7y6si2Zfn+vyntMT37y37w5h/81eXt0E837qb/+mr57f/7zm8/v36lWSyqyHWjQ0u8XMm2ovtn26f70/tdfPn1OyjyQufB0uov32JoN30La564Jemj4RjfAcpniIeJgWreIqmPo6xt22CgsgGI+OZMIVihbkra1c4+zhMoUC7i38WwIHHYsOoRiV/YInchdPIPT1Kd33xb1Oo5+jGYjQF3nA1zgER58Ay86cgDV/MEV38Wb0VdmlEijM+gW4qAUgw33qgXc6b4P5GWl9CnGycQlBQSoLq6ZAfBbysdF0hbPUPZGF1q5DDE+tufewf0YEL8Y1TXQ98c1IYV/3JlXphqDsv0yo4B+B5/ewZSBDUUY1vldqZV6rEI34JehpyrNti7tVpwnHlfMzoPisqrgABWw6YXpsr7sWzGAwtE4824glO5r6FIfZLgmeCda6l1tf/ileWnHH2NvSQEaWwE4hqZbM4bGKw2MYMH5B2lRSI6QYpEcpcSvOwAWdREXo5pvFgesHCw13cYj5iNy6xbdXmo0fMebxuE/bj+G4CF+BfothONEigFuqtoce9qdsYfAwd42ElQW6xet+hzgGOnh6kdSex9kE6ZshgbYKcFjkiY965RabbS9gEYh1ddakFVAGBvbp+HIsU8dnGnfGwNzUqDutd9mmd2OGhGcSWwd/+bs4Tzlsg2Y2GoVOZBOoCHUwwfOOxs7a0IS7JsrkuIOOsgvByBTOaRCE3s2Ln3k03UHtBijYxpkjKU9/FQGGTDNKUuPbjUfpg5HzjIf5n2jzJrY/5jWHiHkcWCaaZfgNNFVxO7T/76s5Oru22lajiz7X9C6nFjqo7y3fXLf21s/2lffHQN7bkS9nuumAvOGav/iSuXHP9cZkHMReV93uG4oWGca+acXmP7+S1eR9ak8ip2Rjm5n7CHqvfb/aeL9ZUPa/fkx0vFrqv47luG3CdNHMF/B9M19Yhqe58Pzz8JgbLSbWz8gGT5hiaWoZd9irl7T8KXMGvWwUaG37f3tAH3bkt7qNpKz2WiDRtfhfBzM4LZF+dUYAStPfJ3deqjh7iNAa1iIxU38bmlfxsHW/0L+4WG13aE+0D8GUGNAffUev410vsoAs7B2P6SAvzP3S4nuMx9ciF8puV9i6GcAmAk+fMJPkew3eGp3+61IGDCcjMffkZUoQN1LW7hkzzQvu0VjDSPR4PowQgrHpk2qBxde6vhWhvP9PmZgepMewru741YZi/Ur3bsQB/0Ig17Nmf8hwDjc2RScLeuqjp2JTxYUYZO0eOGFE1tOUzEJ0R0+NIndkGGTxdrWYx/X2jKaXaW+69Zfv5nv14P4x+KZ+oTZIgzaN4MPxp0NvP8PA6sgj93F2rD+k+lH7dP/vwcEePWDWdq9ZdnP1h3DEWKYItHKRfjh+103A3Juf957XBh05Tv36HwOrgo63WOvkYiR9nO3YuPwzn6b7VxS033Z5VASNuKOhN8P7hVKw+hGYVti953xwKIdMVzTtimIduw3jJcPEq9p6KkbUD9s5Hpc8XdwpWMc7SXvO67ef4kyvaa62KmqhmER3tyGQMfqGA7ZYigSu2bBXbmJ/la96zrG7jqhjvxFnxSDYIHHmacdZ56GAG9MpZ3cGZrv1fsBBHZpiulFmmI4EaUpVkbT1Hy2psukJ/8DuUyhyRpIAAA='}}
for name, info in embedded_files.items():
    content = gzip.decompress(base64.b64decode(info['payload']))
    assert hashlib.sha256(content).hexdigest() == info['sha256']
    (RUNNER_DIR / name).write_bytes(content)

TRAIN_SCRIPT = RUNNER_DIR / '13_last_layer_finetune.py'
COMPARE_SCRIPT = RUNNER_DIR / '14_compare_matched_control.py'
print('Files:', sorted(path.name for path in RUNNER_DIR.iterdir()))


## 4. Run Both Matched Modes

The last-layer mode is checked first so existing completed checkpoints can be
converted into the new comparison format without retraining. The frozen mode
then trains only the external classifier.

The cell prints the full error if either run stops. Rerunning continues from
the saved epoch or fold.


In [ ]:
COMMON = [
    '--kaggle-data-root', str(KAGGLE_DATA_ROOT),
    '--base-output-dir', str(BASE_OUTPUT_DIR),
    '--num-ragas', '5',
    '--tracks-per-raga', '5',
    '--exclude-raga', 'ragamalika',
    '--segments-per-track', '4',
    '--segment-seconds', '30',
    '--batch-size', '1',
    '--gradient-accumulation', '4',
    '--epochs', '8',
    '--patience', '2',
    '--backbone-lr', '1e-5',
    '--classifier-lr', '3e-4',
    '--hidden-dim', '64',
    '--dropout', '0.5',
    '--weight-decay', '0.01',
    '--label-smoothing', '0.1',
    '--baseline-track-accuracy', '0.36',
    '--baseline-track-f1', '0.319',
    '--baseline-train-accuracy', '0.777',
    '--baseline-val-accuracy', '0.490',
]

def stream(command):
    print('\nRunning:', ' '.join(command), flush=True)
    recent = []
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        recent.append(line)
        if len(recent) > 300:
            recent.pop(0)
    code = process.wait()
    if code:
        raise RuntimeError(
            'Run stopped. Read the full log above.\n\n'
            'Last output lines:\n' + ''.join(recent)
        )

stream([
    sys.executable, str(TRAIN_SCRIPT),
    '--mode', 'last_layer',
    '--output-dir', str(LAST_LAYER_DIR),
] + COMMON)

stream([
    sys.executable, str(TRAIN_SCRIPT),
    '--mode', 'frozen',
    '--output-dir', str(FROZEN_DIR),
] + COMMON)

print('\nBoth matched modes completed.')


## 5. Create The Direct Comparison

This cell verifies that every non-trainability setting is identical, compares
the same 25 test recordings, and creates paired fold and confusion plots.


In [ ]:
stream([
    sys.executable,
    str(COMPARE_SCRIPT),
    '--frozen-dir', str(FROZEN_DIR),
    '--finetuned-dir', str(LAST_LAYER_DIR),
    '--output-dir', str(COMPARISON_DIR),
    '--optimized-frozen-reference', '0.36',
])


## 6. Read The Report

The matched result is the valid answer to whether unfreezing layer 12 helped
under this exact training procedure.


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

report_path = COMPARISON_DIR / 'REPORT.md'
metrics_dir = COMPARISON_DIR / 'metrics'
figures_dir = COMPARISON_DIR / 'figures'
assert report_path.exists()

display(Markdown(report_path.read_text(encoding='utf-8')))
display(pd.read_csv(metrics_dir / 'matched_summary.csv'))
display(pd.read_csv(metrics_dir / 'matched_fold_comparison.csv'))
display(json.loads((metrics_dir / 'paired_statistics.json').read_text()))

for name in [
    'matched_overall_comparison.png',
    'matched_fold_comparison.png',
    'matched_track_confusions.png',
]:
    display(Image(filename=str(figures_dir / name)))


## 7. Download

This ZIP is the report to show the mentor. It contains the direct comparison,
fold results, paired per-track predictions, statistics and figures.


In [ ]:
from IPython.display import FileLink, display

result_zip = COMPARISON_DIR / 'mert_matched_control_report.zip'
assert result_zip.exists()
print('Result size:', round(result_zip.stat().st_size / 1e6, 2), 'MB')
display(FileLink(str(result_zip)))
